In [ ]:
# # =========================
# # Figure 3C — SHAP dependence plot(s)
# # 1–2 key features (top by mean |SHAP|), for a microbiome → phenotype model
# # Standalone: loads data, trains LGBM, computes SHAP, saves dependence plots.
# #
# # UPDATE:
# # - Removes the “non-existent / floor” value per-feature BEFORE plotting (fixes the vertical strip)
# # - Uses wider percentile limits + larger padding so points don’t get clipped:
# #     X_PCTL = (0.5, 99.5)
# #     pad = 0.12 * (hi - lo + 1e-9)
# #   (applied to BOTH x and y)
# # =========================

# import os
# import re
# import pickle
# import numpy as np
# import pandas as pd
# import matplotlib as mpl
# import matplotlib.pyplot as plt
# import shap
# from lightgbm import LGBMRegressor

# # -------------------------
# # USER SETTINGS
# # -------------------------
# PHENOTYPE = "bt__triglycerides"
# DF_PATH = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/phenotypes_mb.pkl"
# TARGET_PHENOTYPES_PATH = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/target_phenotypes.pkl"

# OUT_DIR = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/SHAP_plots/phenotypes_predictions/thesis_fig3C"
# os.makedirs(OUT_DIR, exist_ok=True)

# BASE_FEATURES = ["age", "sex"]

# TOP_K = 2                 # set to 1 or 2
# RANDOM_SEED = 0
# BACKGROUND_SIZE = 1000    # for SHAP TreeExplainer background
# SHAP_SAMPLE_SIZE = 20000  # max rows used for SHAP computation/plots (subsampled)

# # If you want to force specific features instead of auto top-|SHAP|:
# FORCE_FEATURES = None     # e.g. ["Faecalibacterium_prausnitzii", "Bifidobacterium_longum"]

# # ---- plot axis control (UPDATED) ----
# X_PCTL = (0.5, 99.5)      # widened from (1,99) so tails aren’t clipped
# Y_PCTL = (0.5, 99.5)
# PAD_FRAC = 0.12           # increased padding

# # ---- “non-existent” value handling ----
# # We infer the floor from the data itself per-feature:
# # If a single value dominates (most common) and sits at the extreme low tail, treat it as the floor.
# DOMINANT_MIN_FRAC = 0.01  # only consider a “floor” if it’s >= 1% of samples
# EXTREME_MIN_Q = 0.02      # and that dominant value is within the bottom 2% of x values

# # -------------------------
# # Helpers
# # -------------------------
# def sanitize_filename(s: str) -> str:
#     return re.sub(r"[^\w\-_. ]", "_", str(s))

# def ensure_numeric_sex(series: pd.Series) -> pd.Series:
#     """Convert sex to 0/1 if it's string-like."""
#     if pd.api.types.is_numeric_dtype(series):
#         return series
#     s = series.astype(str).str.lower().str.strip()
#     mapping = {"m": 1, "male": 1, "man": 1, "1": 1,
#                "f": 0, "female": 0, "woman": 0, "0": 0}
#     return s.map(mapping)

# def subsample_df(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
#     if len(df) <= n:
#         return df
#     return df.sample(n=n, random_state=seed)

# def median_impute(X: pd.DataFrame) -> pd.DataFrame:
#     X2 = X.copy()
#     for col in X2.columns:
#         if X2[col].isna().any():
#             X2[col] = X2[col].fillna(X2[col].median())
#     return X2

# def robust_limits(arr: np.ndarray, pctl=(0.5, 99.5), pad_frac=0.12):
#     """Percentile limits + padding (UPDATED to match requested settings)."""
#     arr = np.asarray(arr)
#     arr = arr[np.isfinite(arr)]
#     if arr.size == 0:
#         return None
#     lo, hi = np.percentile(arr, pctl)
#     if not np.isfinite(lo) or not np.isfinite(hi):
#         return None
#     if hi == lo:
#         # tiny symmetric pad if constant
#         lo -= 1.0
#         hi += 1.0
#     pad = pad_frac * (hi - lo + 1e-9)
#     return lo - pad, hi + pad

# def infer_floor_value(x: np.ndarray):
#     """
#     Infer a per-feature “non-existent/floor” value from x itself:
#     - take most common rounded value (mode-ish)
#     - require it appears in >= DOMINANT_MIN_FRAC of samples
#     - require it lies in the extreme low tail (<= EXTREME_MIN_Q quantile)
#     Returns (floor_value or None, frac).
#     """
#     x = np.asarray(x)
#     x = x[np.isfinite(x)]
#     if x.size < 100:
#         return None, 0.0

#     # Round to stabilize floating comparisons (z-scored values still cluster tightly)
#     xr = np.round(x, 6)
#     vals, counts = np.unique(xr, return_counts=True)
#     k = int(np.argmax(counts))
#     v = float(vals[k])
#     frac = counts[k] / x.size

#     if frac < DOMINANT_MIN_FRAC:
#         return None, frac

#     low_q = float(np.quantile(xr, EXTREME_MIN_Q))
#     if v <= low_q + 1e-12:
#         return v, frac

#     return None, frac

# # -------------------------
# # Load data + build X,y
# # -------------------------
# df = pd.read_pickle(DF_PATH)

# with open(TARGET_PHENOTYPES_PATH, "rb") as f:
#     target_phenotypes = pickle.load(f)

# flat_targets = sum(target_phenotypes, [])
# exclude_cols = set(flat_targets + BASE_FEATURES + ["RegistrationCode"])
# microbial_features = [c for c in df.columns if c not in exclude_cols]

# df_target = df[df[PHENOTYPE].notna()].copy()
# y = df_target[PHENOTYPE].astype(float)

# X = df_target[BASE_FEATURES + microbial_features].copy()

# # make sex numeric if needed
# X["sex"] = ensure_numeric_sex(X["sex"])

# # drop rows with unmapped sex or missing y
# Xy = pd.concat([X, y.rename("y")], axis=1).dropna(subset=["sex", "y"])
# y = Xy["y"]
# X = Xy.drop(columns=["y"])

# # simple imputation
# X = median_impute(X)

# # -------------------------
# # Train LightGBM
# # -------------------------
# n_estimators = 300 if len(y) < 3000 else 800
# learning_rate = 0.05 if len(y) < 3000 else 0.01

# model = LGBMRegressor(
#     objective="regression",
#     n_estimators=n_estimators,
#     learning_rate=learning_rate,
#     max_depth=3,
#     subsample=0.9,
#     colsample_bytree=0.8,
#     min_child_samples=5,
#     n_jobs=8,
#     random_state=RANDOM_SEED,
#     verbose=-1,
# )
# model.fit(X, y)

# # -------------------------
# # Compute SHAP (TreeExplainer) on a subsample
# # -------------------------
# X_used = subsample_df(X, SHAP_SAMPLE_SIZE, RANDOM_SEED)
# bg = subsample_df(X_used, min(BACKGROUND_SIZE, len(X_used)), RANDOM_SEED)

# explainer = shap.TreeExplainer(model, data=bg, feature_perturbation="interventional")
# shap_values = explainer.shap_values(X_used)
# if isinstance(shap_values, list):  # safety
#     shap_values = shap_values[0]
# shap_values = np.asarray(shap_values)

# feature_names = list(X_used.columns)

# # -------------------------
# # Pick 1–2 features for dependence by mean(|SHAP|)
# # -------------------------
# if FORCE_FEATURES is not None:
#     chosen = [f for f in FORCE_FEATURES if f in feature_names]
#     if len(chosen) == 0:
#         raise ValueError("FORCE_FEATURES provided but none found in X columns.")
#     chosen = chosen[:TOP_K]
# else:
#     mean_abs = np.mean(np.abs(shap_values), axis=0)
#     ranked_idx = np.argsort(mean_abs)[::-1]

#     chosen = []
#     for j in ranked_idx:
#         f = feature_names[j]
#         if f in BASE_FEATURES:
#             continue
#         chosen.append(f)
#         if len(chosen) == TOP_K:
#             break

# print("Chosen dependence feature(s):", chosen)

# # -------------------------
# # Plot Figure 3C dependence
# # -------------------------
# mpl.rcParams.update(mpl.rcParamsDefault)
# plt.rcParams.update({"font.size": 12})

# safe_pheno = sanitize_filename(PHENOTYPE)

# for feat in chosen:
#     feat_idx = feature_names.index(feat)

#     # x=feature values, y=SHAP values
#     x = X_used[feat].to_numpy()
#     y_shap = shap_values[:, feat_idx]

#     # ---- infer and remove floor/non-existent value (fixes the strip) ----
#     floor_val, frac = infer_floor_value(x)
#     mask = np.isfinite(x) & np.isfinite(y_shap)
#     if floor_val is not None:
#         mask &= (np.round(x, 6) != np.round(floor_val, 6))
#         print(f"[{feat}] inferred floor={floor_val:.6f} (≈{100*frac:.1f}% of samples) → removed for plotting")
#     else:
#         print(f"[{feat}] no clear floor inferred (dominant value frac={100*frac:.2f}%) → no removal")

#     x_plot = x[mask]
#     y_plot = y_shap[mask]

#     # ---- plotting ----
#     plt.figure(figsize=(5.2, 4.6), dpi=300)
#     plt.scatter(x_plot, y_plot, alpha=0.55, s=10, edgecolor="none")
#     # plt.axhline(0, linewidth=1.0)

#     # ---- UPDATED axis limits: wider percentiles + larger padding ----
#     xlim = robust_limits(x_plot, pctl=X_PCTL, pad_frac=PAD_FRAC)
#     ylim = robust_limits(y_plot, pctl=Y_PCTL, pad_frac=PAD_FRAC)
#     if xlim is not None:
#         plt.xlim(*xlim)
#     if ylim is not None:
#         plt.ylim(*ylim)

#     plt.xlabel(f"{feat} (feature value)")
#     plt.ylabel("SHAP value\n(contribution to predicted phenotype)")
#     plt.title(f"{PHENOTYPE} — SHAP dependence", pad=8)

#     plt.tight_layout()

#     out_png = os.path.join(OUT_DIR, f"{safe_pheno}__dependence__{sanitize_filename(feat)}.png")
#     out_pdf = os.path.join(OUT_DIR, f"{safe_pheno}__dependence__{sanitize_filename(feat)}.pdf")
#     plt.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
#     plt.savefig(out_pdf, dpi=300, bbox_inches="tight", facecolor="white")
#     plt.close()

# print(f"Saved dependence plots to: {OUT_DIR}")


In [ ]:
# # =========================
# # Figure 3C — SHAP dependence plot(s)
# # TOP 20 key features (top by mean |SHAP|), for a microbiome → phenotype model
# # Standalone: loads data, trains LGBM, computes SHAP, saves dependence plots.
# #
# # UPDATE (per your request):
# # - Generates dependence plots for TOP_K = 20 predictors (excludes age/sex)
# # - Keeps your “floor/non-existent value” removal per-feature before plotting
# # - Keeps widened percentile limits + padding to avoid clipping
# # =========================

# import os
# import re
# import pickle
# import numpy as np
# import pandas as pd
# import matplotlib as mpl
# import matplotlib.pyplot as plt
# import shap
# from lightgbm import LGBMRegressor

# # -------------------------
# # USER SETTINGS
# # -------------------------
# PHENOTYPE = "bt__triglycerides"
# DF_PATH = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/phenotypes_mb.pkl"
# TARGET_PHENOTYPES_PATH = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/target_phenotypes.pkl"

# OUT_DIR = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/SHAP_plots/phenotypes_predictions/thesis_fig3C_top20"
# os.makedirs(OUT_DIR, exist_ok=True)

# BASE_FEATURES = ["age", "sex"]

# TOP_K = 20                # <-- UPDATED: top 20 predictors
# RANDOM_SEED = 0
# BACKGROUND_SIZE = 1000    # for SHAP TreeExplainer background
# SHAP_SAMPLE_SIZE = 20000  # max rows used for SHAP computation/plots (subsampled)

# # If you want to force specific features instead of auto top-|SHAP|:
# FORCE_FEATURES = None     # e.g. ["Faecalibacterium_prausnitzii", "Bifidobacterium_longum", ...]

# # ---- plot axis control ----
# X_PCTL = (0.5, 99.5)
# Y_PCTL = (0.5, 99.5)
# PAD_FRAC = 0.12

# # ---- “non-existent” value handling ----
# DOMINANT_MIN_FRAC = 0.01  # only consider a “floor” if it’s >= 1% of samples
# EXTREME_MIN_Q = 0.02      # and that dominant value is within the bottom 2% of x values

# # -------------------------
# # Helpers
# # -------------------------
# def sanitize_filename(s: str) -> str:
#     return re.sub(r"[^\w\-_. ]", "_", str(s))

# def ensure_numeric_sex(series: pd.Series) -> pd.Series:
#     """Convert sex to 0/1 if it's string-like."""
#     if pd.api.types.is_numeric_dtype(series):
#         return series
#     s = series.astype(str).str.lower().str.strip()
#     mapping = {"m": 1, "male": 1, "man": 1, "1": 1,
#                "f": 0, "female": 0, "woman": 0, "0": 0}
#     return s.map(mapping)

# def subsample_df(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
#     if len(df) <= n:
#         return df
#     return df.sample(n=n, random_state=seed)

# def median_impute(X: pd.DataFrame) -> pd.DataFrame:
#     X2 = X.copy()
#     for col in X2.columns:
#         if X2[col].isna().any():
#             X2[col] = X2[col].fillna(X2[col].median())
#     return X2

# def robust_limits(arr: np.ndarray, pctl=(0.5, 99.5), pad_frac=0.12):
#     """Percentile limits + padding."""
#     arr = np.asarray(arr)
#     arr = arr[np.isfinite(arr)]
#     if arr.size == 0:
#         return None
#     lo, hi = np.percentile(arr, pctl)
#     if not np.isfinite(lo) or not np.isfinite(hi):
#         return None
#     if hi == lo:
#         lo -= 1.0
#         hi += 1.0
#     pad = pad_frac * (hi - lo + 1e-9)
#     return lo - pad, hi + pad

# def infer_floor_value(x: np.ndarray):
#     """
#     Infer per-feature “floor” value from x itself:
#     - most common rounded value
#     - appears in >= DOMINANT_MIN_FRAC of samples
#     - lies in extreme low tail (<= EXTREME_MIN_Q quantile)
#     Returns (floor_value or None, frac).
#     """
#     x = np.asarray(x)
#     x = x[np.isfinite(x)]
#     if x.size < 100:
#         return None, 0.0

#     xr = np.round(x, 6)
#     vals, counts = np.unique(xr, return_counts=True)
#     k = int(np.argmax(counts))
#     v = float(vals[k])
#     frac = counts[k] / x.size

#     if frac < DOMINANT_MIN_FRAC:
#         return None, frac

#     low_q = float(np.quantile(xr, EXTREME_MIN_Q))
#     if v <= low_q + 1e-12:
#         return v, frac

#     return None, frac

# # -------------------------
# # Load data + build X,y
# # -------------------------
# df = pd.read_pickle(DF_PATH)

# with open(TARGET_PHENOTYPES_PATH, "rb") as f:
#     target_phenotypes = pickle.load(f)

# flat_targets = sum(target_phenotypes, [])
# exclude_cols = set(flat_targets + BASE_FEATURES + ["RegistrationCode"])
# microbial_features = [c for c in df.columns if c not in exclude_cols]

# df_target = df[df[PHENOTYPE].notna()].copy()
# y = df_target[PHENOTYPE].astype(float)

# X = df_target[BASE_FEATURES + microbial_features].copy()

# # make sex numeric if needed
# X["sex"] = ensure_numeric_sex(X["sex"])

# # drop rows with unmapped sex or missing y
# Xy = pd.concat([X, y.rename("y")], axis=1).dropna(subset=["sex", "y"])
# y = Xy["y"]
# X = Xy.drop(columns=["y"])

# # simple imputation
# X = median_impute(X)

# # -------------------------
# # Train LightGBM
# # -------------------------
# n_estimators = 300 if len(y) < 3000 else 800
# learning_rate = 0.05 if len(y) < 3000 else 0.01

# model = LGBMRegressor(
#     objective="regression",
#     n_estimators=n_estimators,
#     learning_rate=learning_rate,
#     max_depth=3,
#     subsample=0.9,
#     colsample_bytree=0.8,
#     min_child_samples=5,
#     n_jobs=8,
#     random_state=RANDOM_SEED,
#     verbose=-1,
# )
# model.fit(X, y)

# # -------------------------
# # Compute SHAP (TreeExplainer) on a subsample
# # -------------------------
# X_used = subsample_df(X, SHAP_SAMPLE_SIZE, RANDOM_SEED)
# bg = subsample_df(X_used, min(BACKGROUND_SIZE, len(X_used)), RANDOM_SEED)

# explainer = shap.TreeExplainer(model, data=bg, feature_perturbation="interventional")
# shap_values = explainer.shap_values(X_used)
# if isinstance(shap_values, list):
#     shap_values = shap_values[0]
# shap_values = np.asarray(shap_values)

# feature_names = list(X_used.columns)

# # -------------------------
# # Pick TOP_K features for dependence by mean(|SHAP|)
# # -------------------------
# if FORCE_FEATURES is not None:
#     chosen = [f for f in FORCE_FEATURES if f in feature_names]
#     if len(chosen) == 0:
#         raise ValueError("FORCE_FEATURES provided but none found in X columns.")
#     chosen = chosen[:TOP_K]
# else:
#     mean_abs = np.mean(np.abs(shap_values), axis=0)
#     ranked_idx = np.argsort(mean_abs)[::-1]

#     chosen = []
#     for j in ranked_idx:
#         f = feature_names[j]
#         if f in BASE_FEATURES:
#             continue
#         chosen.append(f)
#         if len(chosen) == TOP_K:
#             break

# print(f"Chosen dependence feature(s) (TOP {len(chosen)}):")
# for k, f in enumerate(chosen, 1):
#     print(f"  {k:02d}. {f}")

# # -------------------------
# # Plot dependence for TOP_K features
# # -------------------------
# mpl.rcParams.update(mpl.rcParamsDefault)
# plt.rcParams.update({"font.size": 12})

# safe_pheno = sanitize_filename(PHENOTYPE)

# # Save a quick manifest of selected features
# with open(os.path.join(OUT_DIR, f"{safe_pheno}__top{len(chosen)}_features.txt"), "w") as f:
#     f.write("\n".join(chosen) + "\n")

# for rank, feat in enumerate(chosen, start=1):
#     feat_idx = feature_names.index(feat)

#     x = X_used[feat].to_numpy()
#     y_shap = shap_values[:, feat_idx]

#     # ---- infer and remove floor/non-existent value (strip) ----
#     floor_val, frac = infer_floor_value(x)
#     mask = np.isfinite(x) & np.isfinite(y_shap)
#     if floor_val is not None:
#         mask &= (np.round(x, 6) != np.round(floor_val, 6))
#         print(f"[{rank:02d}/{len(chosen)}] {feat}: floor={floor_val:.6f} (~{100*frac:.1f}%) removed for plotting")
#     else:
#         print(f"[{rank:02d}/{len(chosen)}] {feat}: no clear floor (dominant frac={100*frac:.2f}%)")

#     x_plot = x[mask]
#     y_plot = y_shap[mask]

#     # ---- plotting ----
#     plt.figure(figsize=(5.2, 4.6), dpi=300)
#     plt.scatter(x_plot, y_plot, alpha=0.55, s=10, edgecolor="none")
#     # plt.axhline(0, linewidth=1.0)

#     # limits with padding
#     xlim = robust_limits(x_plot, pctl=X_PCTL, pad_frac=PAD_FRAC)
#     ylim = robust_limits(y_plot, pctl=Y_PCTL, pad_frac=PAD_FRAC)
#     if xlim is not None:
#         plt.xlim(*xlim)
#     if ylim is not None:
#         plt.ylim(*ylim)

#     plt.xlabel(f"{feat} (feature value)")
#     plt.ylabel("SHAP value\n(contribution to predicted phenotype)")
#     plt.title(f"{PHENOTYPE} — SHAP dependence", pad=8)

#     plt.tight_layout()

#     safe_feat = sanitize_filename(feat)
#     out_png = os.path.join(OUT_DIR, f"{safe_pheno}__rank{rank:02d}__dependence__{safe_feat}.png")
#     out_pdf = os.path.join(OUT_DIR, f"{safe_pheno}__rank{rank:02d}__dependence__{safe_feat}.pdf")
#     plt.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
#     plt.savefig(out_pdf, dpi=300, bbox_inches="tight", facecolor="white")
#     plt.close()

# print(f"Saved TOP {len(chosen)} dependence plots to: {OUT_DIR}")


In [ ]:
# # =========================
# # Figure 3C — SHAP dependence for ONE feature (Bifidobacterium longum)
# # Standalone: loads data, trains LGBM, computes SHAP, saves ONE dependence plot.
# #
# # Includes:
# # - Optional removal of per-feature "floor/non-existent" value (fix vertical strip)
# # - Padded percentile axis limits (avoid clipped points)
# # - Optional coloring by a covariate/interaction feature (e.g., "age" or "Otoolea fessa")
# # - Running-median trend line
# # - Marginal distributions (top x, right y)
# # - Secondary top x-axis: cohort percentile (makes z-score easier to interpret)
# # =========================

# import os
# import re
# import pickle
# import numpy as np
# import pandas as pd
# import matplotlib as mpl
# import matplotlib.pyplot as plt
# import shap
# from lightgbm import LGBMRegressor
# from matplotlib.gridspec import GridSpec

# # -------------------------
# # USER SETTINGS
# # -------------------------
# PHENOTYPE = "bt__triglycerides"
# MAIN_FEATURE = "Bifidobacterium longum"  # must match EXACTLY a column in X

# DF_PATH = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/phenotypes_mb.pkl"
# TARGET_PHENOTYPES_PATH = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/target_phenotypes.pkl"

# OUT_DIR = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/SHAP_plots/phenotypes_predictions/thesis_fig3C"
# os.makedirs(OUT_DIR, exist_ok=True)

# BASE_FEATURES = ["age", "sex"]

# # SHAP + compute settings
# RANDOM_SEED = 0
# BACKGROUND_SIZE = 1000
# SHAP_SAMPLE_SIZE = 20000

# # Plot settings
# FIG_W, FIG_H = 6.2, 5.2
# FONT = 12

# # Color points by another feature (set None for no coloring)
# # Good options: "age", "sex" (but sex is better as 2 panels), or another taxon.
# COLOR_BY = "age"   # or "auto" (not used here), or None

# # Axis padding / clipping prevention
# X_PCTL = (0.5, 99.5)
# Y_PCTL = (0.5, 99.5)
# PAD_FRAC = 0.12

# # Floor/non-existent value removal to fix vertical strip
# REMOVE_FLOOR = True
# DOMINANT_MIN_FRAC = 0.01
# EXTREME_MIN_Q = 0.02

# # -------------------------
# # Helpers
# # -------------------------
# def sanitize_filename(s: str) -> str:
#     return re.sub(r"[^\w\-_. ]", "_", str(s))

# def ensure_numeric_sex(series: pd.Series) -> pd.Series:
#     if pd.api.types.is_numeric_dtype(series):
#         return series
#     s = series.astype(str).str.lower().str.strip()
#     mapping = {"m": 1, "male": 1, "man": 1, "1": 1,
#                "f": 0, "female": 0, "woman": 0, "0": 0}
#     return s.map(mapping)

# def subsample_df(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
#     if len(df) <= n:
#         return df
#     return df.sample(n=n, random_state=seed)

# def median_impute(X: pd.DataFrame) -> pd.DataFrame:
#     X2 = X.copy()
#     for col in X2.columns:
#         if X2[col].isna().any():
#             X2[col] = X2[col].fillna(X2[col].median())
#     return X2

# def robust_limits(arr: np.ndarray, pctl=(0.5, 99.5), pad_frac=0.12):
#     arr = np.asarray(arr)
#     arr = arr[np.isfinite(arr)]
#     if arr.size == 0:
#         return None
#     lo, hi = np.percentile(arr, pctl)
#     if not np.isfinite(lo) or not np.isfinite(hi):
#         return None
#     if hi == lo:
#         lo -= 1.0
#         hi += 1.0
#     pad = pad_frac * (hi - lo + 1e-9)
#     return lo - pad, hi + pad

# def infer_floor_value(x: np.ndarray):
#     x = np.asarray(x)
#     x = x[np.isfinite(x)]
#     if x.size < 100:
#         return None, 0.0

#     xr = np.round(x, 6)
#     vals, counts = np.unique(xr, return_counts=True)
#     k = int(np.argmax(counts))
#     v = float(vals[k])
#     frac = counts[k] / x.size

#     if frac < DOMINANT_MIN_FRAC:
#         return None, frac

#     low_q = float(np.quantile(xr, EXTREME_MIN_Q))
#     if v <= low_q + 1e-12:
#         return v, frac

#     return None, frac

# def running_median_line(x, y, n_bins=45, min_per_bin=35):
#     x = np.asarray(x); y = np.asarray(y)
#     ok = np.isfinite(x) & np.isfinite(y)
#     x = x[ok]; y = y[ok]
#     if x.size < 200:
#         return None

#     edges = np.linspace(np.nanpercentile(x, 1), np.nanpercentile(x, 99), n_bins + 1)
#     mids, meds = [], []
#     for a, b in zip(edges[:-1], edges[1:]):
#         m = (x >= a) & (x < b)
#         if m.sum() >= min_per_bin:
#             mids.append((a + b) / 2)
#             meds.append(np.median(y[m]))
#     if len(mids) < 5:
#         return None
#     return np.array(mids), np.array(meds)

# # -------------------------
# # Load data + build X,y
# # -------------------------
# df = pd.read_pickle(DF_PATH)

# with open(TARGET_PHENOTYPES_PATH, "rb") as f:
#     target_phenotypes = pickle.load(f)

# flat_targets = sum(target_phenotypes, [])
# exclude_cols = set(flat_targets + BASE_FEATURES + ["RegistrationCode"])
# microbial_features = [c for c in df.columns if c not in exclude_cols]

# df_target = df[df[PHENOTYPE].notna()].copy()
# y = df_target[PHENOTYPE].astype(float)

# X = df_target[BASE_FEATURES + microbial_features].copy()
# X["sex"] = ensure_numeric_sex(X["sex"])

# # Drop rows with unmapped sex or missing y
# Xy = pd.concat([X, y.rename("y")], axis=1).dropna(subset=["sex", "y"])
# y = Xy["y"]
# X = Xy.drop(columns=["y"])

# # Impute
# X = median_impute(X)

# if MAIN_FEATURE not in X.columns:
#     raise ValueError(f"MAIN_FEATURE='{MAIN_FEATURE}' not found in X columns. "
#                      f"Example columns: {list(X.columns)[:10]} ...")

# # -------------------------
# # Train LightGBM
# # -------------------------
# n_estimators = 300 if len(y) < 3000 else 800
# learning_rate = 0.05 if len(y) < 3000 else 0.01

# model = LGBMRegressor(
#     objective="regression",
#     n_estimators=n_estimators,
#     learning_rate=learning_rate,
#     max_depth=3,
#     subsample=0.9,
#     colsample_bytree=0.8,
#     min_child_samples=5,
#     n_jobs=8,
#     random_state=RANDOM_SEED,
#     verbose=-1,
# )
# model.fit(X, y)

# # -------------------------
# # Compute SHAP on subsample
# # -------------------------
# X_used = subsample_df(X, SHAP_SAMPLE_SIZE, RANDOM_SEED)
# bg = subsample_df(X_used, min(BACKGROUND_SIZE, len(X_used)), RANDOM_SEED)

# explainer = shap.TreeExplainer(model, data=bg, feature_perturbation="interventional")
# shap_values = explainer.shap_values(X_used)
# if isinstance(shap_values, list):
#     shap_values = shap_values[0]
# shap_values = np.asarray(shap_values)

# # -------------------------
# # Plot dependence for ONE feature
# # -------------------------
# mpl.rcParams.update(mpl.rcParamsDefault)
# plt.rcParams.update({"font.size": FONT})

# feat_idx = list(X_used.columns).index(MAIN_FEATURE)
# x = X_used[MAIN_FEATURE].to_numpy()
# y_shap = shap_values[:, feat_idx]

# mask = np.isfinite(x) & np.isfinite(y_shap)

# # Remove floor strip if detected
# if REMOVE_FLOOR:
#     floor_val, frac = infer_floor_value(x)
#     if floor_val is not None:
#         mask &= (np.round(x, 6) != np.round(floor_val, 6))
#         print(f"[{MAIN_FEATURE}] removed inferred floor={floor_val:.6f} (~{100*frac:.1f}%) for plotting")

# x = x[mask]
# y_plot = y_shap[mask]

# # Color-by feature
# c = None
# c_label = None
# if COLOR_BY is not None:
#     if COLOR_BY not in X_used.columns:
#         raise ValueError(f"COLOR_BY='{COLOR_BY}' not found in X_used columns.")
#     c = X_used.loc[X_used.index[mask], COLOR_BY].to_numpy()
#     c_label = COLOR_BY

# # Layout with marginals
# fig = plt.figure(figsize=(FIG_W, FIG_H), dpi=300)
# gs = GridSpec(2, 2, width_ratios=[5.0, 1.4], height_ratios=[1.3, 5.0],
#               wspace=0.05, hspace=0.05)

# ax_top = fig.add_subplot(gs[0, 0])
# ax_main = fig.add_subplot(gs[1, 0])
# ax_right = fig.add_subplot(gs[1, 1])

# # Main scatter
# if c is None:
#     sc = ax_main.scatter(x, y_plot, s=14, alpha=0.55, edgecolor="none")
# else:
#     sc = ax_main.scatter(x, y_plot, c=c, s=14, alpha=0.6, edgecolor="none", cmap="coolwarm")
#     cb = fig.colorbar(sc, ax=ax_main, fraction=0.04, pad=0.02)
#     if c_label == "age":
#         cb.set_label("Age (years)", rotation=90)
#     else:
#         cb.set_label(f"{c_label} (feature value)", rotation=90)

# # ax_main.axhline(0, linewidth=1.0)

# # Trend line
# rm = running_median_line(x, y_plot, n_bins=45, min_per_bin=35)
# if rm is not None:
#     xm, ym = rm
#     ax_main.plot(xm, ym, linewidth=2.2)

# # Robust padded limits
# xlim = robust_limits(x, pctl=X_PCTL, pad_frac=PAD_FRAC)
# ylim = robust_limits(y_plot, pctl=Y_PCTL, pad_frac=PAD_FRAC)
# if xlim: ax_main.set_xlim(*xlim)
# if ylim: ax_main.set_ylim(*ylim)

# # Labels
# ax_main.set_xlabel(f"{MAIN_FEATURE} (z-score)")
# ax_main.set_ylabel("SHAP value")
# # ax_main.set_title(f"{PHENOTYPE} — SHAP dependence", pad=10)

# # Marginals
# ax_top.hist(x, bins=40, density=True)
# ax_top.set_xlim(ax_main.get_xlim())
# ax_top.axis("off")

# ax_right.hist(y_plot, bins=40, density=True, orientation="horizontal")
# ax_right.set_ylim(ax_main.get_ylim())
# ax_right.axis("off")

# # Secondary top x-axis: percentile mapping
# xs = np.sort(x)
# ps = np.linspace(0, 100, len(xs))

# def x_to_pct(v):
#     return np.interp(v, xs, ps)

# def pct_to_x(p):
#     return np.interp(p, ps, xs)

# secax = ax_main.secondary_xaxis("top", functions=(x_to_pct, pct_to_x))
# secax.set_xlabel("Abundance percentile in cohort")
# secax.set_xticks([10, 25, 50, 75, 90])

# fig.tight_layout()

# safe_pheno = sanitize_filename(PHENOTYPE)
# safe_feat = sanitize_filename(MAIN_FEATURE)
# out_png = os.path.join(OUT_DIR, f"{safe_pheno}__dependence__{safe_feat}.png")
# out_pdf = os.path.join(OUT_DIR, f"{safe_pheno}__dependence__{safe_feat}.pdf")

# fig.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
# fig.savefig(out_pdf, dpi=300, bbox_inches="tight", facecolor="white")
# plt.close(fig)

# print(f"Saved:\n  {out_png}\n  {out_pdf}")


In [ ]:
# # =========================
# # Figure 3C — SHAP dependence for ONE feature (Bifidobacterium longum)
# # Standalone: loads data, trains LGBM, computes SHAP, saves ONE dependence plot.
# #
# # Includes:
# # - Optional removal of per-feature "floor/non-existent" value (fix vertical strip)
# # - Padded percentile axis limits (avoid clipped points)
# # - Optional coloring by a covariate/interaction feature (e.g., "age" or "Otoolea fessa")
# # - Running-median trend line
# # - Marginal distributions (top x, right y)
# # - Secondary top x-axis: cohort percentile (makes z-score easier to interpret)
# # =========================

# import os
# import re
# import pickle
# import numpy as np
# import pandas as pd
# import matplotlib as mpl
# import matplotlib.pyplot as plt
# import shap
# from lightgbm import LGBMRegressor
# from matplotlib.gridspec import GridSpec

# # -------------------------
# # USER SETTINGS
# # -------------------------
# PHENOTYPE = "bt__triglycerides"
# MAIN_FEATURE = "Bifidobacterium longum"  # must match EXACTLY a column in X

# DF_PATH = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/phenotypes_mb.pkl"
# TARGET_PHENOTYPES_PATH = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/target_phenotypes.pkl"

# OUT_DIR = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/SHAP_plots/phenotypes_predictions/thesis_fig3C"
# os.makedirs(OUT_DIR, exist_ok=True)

# BASE_FEATURES = ["age", "sex"]

# # SHAP + compute settings
# RANDOM_SEED = 0
# BACKGROUND_SIZE = 1000
# SHAP_SAMPLE_SIZE = 20000

# # Plot settings
# FIG_W, FIG_H = 6.2, 5.2
# FONT = 12

# # Color points by another feature (set None for no coloring)
# # Good options: "age", "sex" (but sex is better as 2 panels), or another taxon.
# COLOR_BY = "age"   # or None

# # Axis padding / clipping prevention
# X_PCTL = (0.5, 99.5)
# Y_PCTL = (0.5, 99.5)
# PAD_FRAC = 0.12

# # Floor/non-existent value removal to fix vertical strip
# REMOVE_FLOOR = True
# DOMINANT_MIN_FRAC = 0.01
# EXTREME_MIN_Q = 0.02

# # -------------------------
# # Helpers
# # -------------------------
# def sanitize_filename(s: str) -> str:
#     return re.sub(r"[^\w\-_. ]", "_", str(s))

# def ensure_numeric_sex(series: pd.Series) -> pd.Series:
#     if pd.api.types.is_numeric_dtype(series):
#         return series
#     s = series.astype(str).str.lower().str.strip()
#     mapping = {"m": 1, "male": 1, "man": 1, "1": 1,
#                "f": 0, "female": 0, "woman": 0, "0": 0}
#     return s.map(mapping)

# def subsample_df(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
#     if len(df) <= n:
#         return df
#     return df.sample(n=n, random_state=seed)

# def median_impute(X: pd.DataFrame) -> pd.DataFrame:
#     X2 = X.copy()
#     for col in X2.columns:
#         if X2[col].isna().any():
#             X2[col] = X2[col].fillna(X2[col].median())
#     return X2

# def robust_limits(arr: np.ndarray, pctl=(0.5, 99.5), pad_frac=0.12):
#     arr = np.asarray(arr)
#     arr = arr[np.isfinite(arr)]
#     if arr.size == 0:
#         return None
#     lo, hi = np.percentile(arr, pctl)
#     if not np.isfinite(lo) or not np.isfinite(hi):
#         return None
#     if hi == lo:
#         lo -= 1.0
#         hi += 1.0
#     pad = pad_frac * (hi - lo + 1e-9)
#     return lo - pad, hi + pad

# def infer_floor_value(x: np.ndarray):
#     x = np.asarray(x)
#     x = x[np.isfinite(x)]
#     if x.size < 100:
#         return None, 0.0

#     xr = np.round(x, 6)
#     vals, counts = np.unique(xr, return_counts=True)
#     k = int(np.argmax(counts))
#     v = float(vals[k])
#     frac = counts[k] / x.size

#     if frac < DOMINANT_MIN_FRAC:
#         return None, frac

#     low_q = float(np.quantile(xr, EXTREME_MIN_Q))
#     if v <= low_q + 1e-12:
#         return v, frac

#     return None, frac

# def running_median_line(x, y, n_bins=45, min_per_bin=35):
#     x = np.asarray(x); y = np.asarray(y)
#     ok = np.isfinite(x) & np.isfinite(y)
#     x = x[ok]; y = y[ok]
#     if x.size < 200:
#         return None

#     edges = np.linspace(np.nanpercentile(x, 1), np.nanpercentile(x, 99), n_bins + 1)
#     mids, meds = [], []
#     for a, b in zip(edges[:-1], edges[1:]):
#         m = (x >= a) & (x < b)
#         if m.sum() >= min_per_bin:
#             mids.append((a + b) / 2)
#             meds.append(np.median(y[m]))
#     if len(mids) < 5:
#         return None
#     return np.array(mids), np.array(meds)

# # -------------------------
# # Load data + build X,y
# # -------------------------
# df = pd.read_pickle(DF_PATH)

# with open(TARGET_PHENOTYPES_PATH, "rb") as f:
#     target_phenotypes = pickle.load(f)

# flat_targets = sum(target_phenotypes, [])
# exclude_cols = set(flat_targets + BASE_FEATURES + ["RegistrationCode"])
# microbial_features = [c for c in df.columns if c not in exclude_cols]

# df_target = df[df[PHENOTYPE].notna()].copy()
# y = df_target[PHENOTYPE].astype(float)

# X = df_target[BASE_FEATURES + microbial_features].copy()
# X["sex"] = ensure_numeric_sex(X["sex"])

# # Drop rows with unmapped sex or missing y
# Xy = pd.concat([X, y.rename("y")], axis=1).dropna(subset=["sex", "y"])
# y = Xy["y"]
# X = Xy.drop(columns=["y"])

# # Impute
# X = median_impute(X)

# if MAIN_FEATURE not in X.columns:
#     raise ValueError(
#         f"MAIN_FEATURE='{MAIN_FEATURE}' not found in X columns. "
#         f"Example columns: {list(X.columns)[:10]} ..."
#     )

# # -------------------------
# # Train LightGBM
# # -------------------------
# n_estimators = 300 if len(y) < 3000 else 800
# learning_rate = 0.05 if len(y) < 3000 else 0.01

# model = LGBMRegressor(
#     objective="regression",
#     n_estimators=n_estimators,
#     learning_rate=learning_rate,
#     max_depth=3,
#     subsample=0.9,
#     colsample_bytree=0.8,
#     min_child_samples=5,
#     n_jobs=8,
#     random_state=RANDOM_SEED,
#     verbose=-1,
# )
# model.fit(X, y)

# # -------------------------
# # Compute SHAP on subsample
# # -------------------------
# X_used = subsample_df(X, SHAP_SAMPLE_SIZE, RANDOM_SEED)
# bg = subsample_df(X_used, min(BACKGROUND_SIZE, len(X_used)), RANDOM_SEED)

# explainer = shap.TreeExplainer(model, data=bg, feature_perturbation="interventional")
# shap_values = explainer.shap_values(X_used)
# if isinstance(shap_values, list):
#     shap_values = shap_values[0]
# shap_values = np.asarray(shap_values)

# # -------------------------
# # Plot dependence for ONE feature
# # -------------------------
# mpl.rcParams.update(mpl.rcParamsDefault)
# plt.rcParams.update({"font.size": FONT})

# feat_idx = list(X_used.columns).index(MAIN_FEATURE)
# x = X_used[MAIN_FEATURE].to_numpy()
# y_shap = shap_values[:, feat_idx]

# mask = np.isfinite(x) & np.isfinite(y_shap)

# # Remove floor strip if detected
# if REMOVE_FLOOR:
#     floor_val, frac = infer_floor_value(x)
#     if floor_val is not None:
#         mask &= (np.round(x, 6) != np.round(floor_val, 6))
#         print(f"[{MAIN_FEATURE}] removed inferred floor={floor_val:.6f} (~{100*frac:.1f}%) for plotting")

# x = x[mask]
# y_plot = y_shap[mask]

# # Color-by feature
# c = None
# c_label = None
# if COLOR_BY is not None:
#     if COLOR_BY not in X_used.columns:
#         raise ValueError(f"COLOR_BY='{COLOR_BY}' not found in X_used columns.")
#     c = X_used.loc[X_used.index[mask], COLOR_BY].to_numpy()
#     c_label = COLOR_BY

# # Layout with marginals
# fig = plt.figure(figsize=(FIG_W, FIG_H), dpi=300)
# gs = GridSpec(
#     2, 2,
#     width_ratios=[5.0, 1.4],
#     height_ratios=[1.3, 5.0],
#     wspace=0.05, hspace=0.05
# )

# ax_top = fig.add_subplot(gs[0, 0])
# ax_main = fig.add_subplot(gs[1, 0])
# ax_right = fig.add_subplot(gs[1, 1])

# # Main scatter
# if c is None:
#     sc = ax_main.scatter(x, y_plot, s=14, alpha=0.55, edgecolor="none")
# else:
#     sc = ax_main.scatter(x, y_plot, c=c, s=14, alpha=0.6, edgecolor="none", cmap="coolwarm")
#     cb = fig.colorbar(sc, ax=ax_main, fraction=0.04, pad=0.02)
#     if c_label == "age":
#         cb.set_label("Age (years)", rotation=90)
#     else:
#         cb.set_label(f"{c_label} (feature value)", rotation=90)

# # Trend line
# rm = running_median_line(x, y_plot, n_bins=45, min_per_bin=35)
# if rm is not None:
#     xm, ym = rm
#     ax_main.plot(xm, ym, linewidth=2.2)

# # Robust padded limits
# xlim = robust_limits(x, pctl=X_PCTL, pad_frac=PAD_FRAC)
# ylim = robust_limits(y_plot, pctl=Y_PCTL, pad_frac=PAD_FRAC)
# if xlim:
#     ax_main.set_xlim(*xlim)
# if ylim:
#     ax_main.set_ylim(*ylim)

# # Labels (minimal thesis style)
# ax_main.set_xlabel("B. longum abundance (z-score)")
# ax_main.set_ylabel("SHAP value for triglycerides")

# # Marginals
# ax_top.hist(x, bins=40, density=True)
# ax_top.set_xlim(ax_main.get_xlim())
# ax_top.axis("off")

# ax_right.hist(y_plot, bins=40, density=True, orientation="horizontal")
# ax_right.set_ylim(ax_main.get_ylim())
# ax_right.axis("off")

# # Secondary top x-axis: percentile mapping (minimal)
# xs = np.sort(x)
# ps = np.linspace(0, 100, len(xs))

# def x_to_pct(v):
#     return np.interp(v, xs, ps)

# def pct_to_x(p):
#     return np.interp(p, ps, xs)

# secax = ax_main.secondary_xaxis("top", functions=(x_to_pct, pct_to_x))
# secax.set_xlabel("Percentile")
# secax.set_xticks([10, 50, 90])

# fig.tight_layout()

# safe_pheno = sanitize_filename(PHENOTYPE)
# safe_feat = sanitize_filename(MAIN_FEATURE)
# out_png = os.path.join(OUT_DIR, f"{safe_pheno}__dependence__{safe_feat}.png")
# out_pdf = os.path.join(OUT_DIR, f"{safe_pheno}__dependence__{safe_feat}.pdf")

# fig.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
# fig.savefig(out_pdf, dpi=300, bbox_inches="tight", facecolor="white")
# plt.close(fig)

# print(f"Saved:\n  {out_png}\n  {out_pdf}")


In [ ]:
# # =========================
# # Figure 3C — SHAP dependence for ONE feature (Bifidobacterium longum)
# # Standalone: loads data, trains LGBM, computes SHAP, saves ONE dependence plot.
# #
# # Includes:
# # - Optional removal of per-feature "floor/non-existent" value (fix vertical strip)
# # - Padded percentile axis limits (avoid clipped points)
# # - Optional coloring by a covariate/interaction feature (e.g., "age" or "Otoolea fessa")
# # - Running-median trend line
# # - Marginal distributions (top x, right y)
# # - Secondary top x-axis: cohort percentile (makes z-score easier to interpret)
# # =========================

# import os
# import re
# import pickle
# import numpy as np
# import pandas as pd
# import matplotlib as mpl
# import matplotlib.pyplot as plt
# import shap
# from lightgbm import LGBMRegressor
# from matplotlib.gridspec import GridSpec

# # -------------------------
# # USER SETTINGS
# # -------------------------
# PHENOTYPE = "bt__triglycerides"
# MAIN_FEATURE = "Bifidobacterium longum"  # must match EXACTLY a column in X

# DF_PATH = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/phenotypes_mb.pkl"
# TARGET_PHENOTYPES_PATH = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/target_phenotypes.pkl"

# OUT_DIR = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/SHAP_plots/phenotypes_predictions/thesis_fig3C"
# os.makedirs(OUT_DIR, exist_ok=True)

# BASE_FEATURES = ["age", "sex"]

# # SHAP + compute settings
# RANDOM_SEED = 0
# BACKGROUND_SIZE = 1000
# SHAP_SAMPLE_SIZE = 20000

# # Plot settings
# FIG_W, FIG_H = 6.2, 5.2
# FONT = 12

# # Color points by another feature (set None for no coloring)
# COLOR_BY = "age"   # or None

# # Axis padding / clipping prevention
# X_PCTL = (0.5, 99.5)
# Y_PCTL = (0.5, 99.5)
# PAD_FRAC = 0.12

# # Floor/non-existent value removal to fix vertical strip
# REMOVE_FLOOR = True
# DOMINANT_MIN_FRAC = 0.01
# EXTREME_MIN_Q = 0.02

# # -------------------------
# # Helpers
# # -------------------------
# def sanitize_filename(s: str) -> str:
#     return re.sub(r"[^\w\-_. ]", "_", str(s))

# def ensure_numeric_sex(series: pd.Series) -> pd.Series:
#     if pd.api.types.is_numeric_dtype(series):
#         return series
#     s = series.astype(str).str.lower().str.strip()
#     mapping = {"m": 1, "male": 1, "man": 1, "1": 1,
#                "f": 0, "female": 0, "woman": 0, "0": 0}
#     return s.map(mapping)

# def subsample_df(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
#     if len(df) <= n:
#         return df
#     return df.sample(n=n, random_state=seed)

# def median_impute(X: pd.DataFrame) -> pd.DataFrame:
#     X2 = X.copy()
#     for col in X2.columns:
#         if X2[col].isna().any():
#             X2[col] = X2[col].fillna(X2[col].median())
#     return X2

# def robust_limits(arr: np.ndarray, pctl=(0.5, 99.5), pad_frac=0.12):
#     arr = np.asarray(arr)
#     arr = arr[np.isfinite(arr)]
#     if arr.size == 0:
#         return None
#     lo, hi = np.percentile(arr, pctl)
#     if not np.isfinite(lo) or not np.isfinite(hi):
#         return None
#     if hi == lo:
#         lo -= 1.0
#         hi += 1.0
#     pad = pad_frac * (hi - lo + 1e-9)
#     return lo - pad, hi + pad

# def infer_floor_value(x: np.ndarray):
#     x = np.asarray(x)
#     x = x[np.isfinite(x)]
#     if x.size < 100:
#         return None, 0.0

#     xr = np.round(x, 6)
#     vals, counts = np.unique(xr, return_counts=True)
#     k = int(np.argmax(counts))
#     v = float(vals[k])
#     frac = counts[k] / x.size

#     if frac < DOMINANT_MIN_FRAC:
#         return None, frac

#     low_q = float(np.quantile(xr, EXTREME_MIN_Q))
#     if v <= low_q + 1e-12:
#         return v, frac

#     return None, frac

# def running_median_line(x, y, n_bins=45, min_per_bin=35):
#     x = np.asarray(x); y = np.asarray(y)
#     ok = np.isfinite(x) & np.isfinite(y)
#     x = x[ok]; y = y[ok]
#     if x.size < 200:
#         return None

#     edges = np.linspace(np.nanpercentile(x, 1), np.nanpercentile(x, 99), n_bins + 1)
#     mids, meds = [], []
#     for a, b in zip(edges[:-1], edges[1:]):
#         m = (x >= a) & (x < b)
#         if m.sum() >= min_per_bin:
#             mids.append((a + b) / 2)
#             meds.append(np.median(y[m]))
#     if len(mids) < 5:
#         return None
#     return np.array(mids), np.array(meds)

# # -------------------------
# # Load data + build X,y
# # -------------------------
# df = pd.read_pickle(DF_PATH)

# with open(TARGET_PHENOTYPES_PATH, "rb") as f:
#     target_phenotypes = pickle.load(f)

# flat_targets = sum(target_phenotypes, [])
# exclude_cols = set(flat_targets + BASE_FEATURES + ["RegistrationCode"])
# microbial_features = [c for c in df.columns if c not in exclude_cols]

# df_target = df[df[PHENOTYPE].notna()].copy()
# y = df_target[PHENOTYPE].astype(float)

# X = df_target[BASE_FEATURES + microbial_features].copy()
# X["sex"] = ensure_numeric_sex(X["sex"])

# Xy = pd.concat([X, y.rename("y")], axis=1).dropna(subset=["sex", "y"])
# y = Xy["y"]
# X = Xy.drop(columns=["y"])

# X = median_impute(X)

# if MAIN_FEATURE not in X.columns:
#     raise ValueError(
#         f"MAIN_FEATURE='{MAIN_FEATURE}' not found in X columns. "
#         f"Example columns: {list(X.columns)[:10]} ..."
#     )

# # -------------------------
# # Train LightGBM
# # -------------------------
# n_estimators = 300 if len(y) < 3000 else 800
# learning_rate = 0.05 if len(y) < 3000 else 0.01

# model = LGBMRegressor(
#     objective="regression",
#     n_estimators=n_estimators,
#     learning_rate=learning_rate,
#     max_depth=3,
#     subsample=0.9,
#     colsample_bytree=0.8,
#     min_child_samples=5,
#     n_jobs=8,
#     random_state=RANDOM_SEED,
#     verbose=-1,
# )
# model.fit(X, y)

# # -------------------------
# # Compute SHAP on subsample
# # -------------------------
# X_used = subsample_df(X, SHAP_SAMPLE_SIZE, RANDOM_SEED)
# bg = subsample_df(X_used, min(BACKGROUND_SIZE, len(X_used)), RANDOM_SEED)

# explainer = shap.TreeExplainer(model, data=bg, feature_perturbation="interventional")
# shap_values = explainer.shap_values(X_used)
# if isinstance(shap_values, list):
#     shap_values = shap_values[0]
# shap_values = np.asarray(shap_values)

# # -------------------------
# # Plot dependence for ONE feature
# # -------------------------
# mpl.rcParams.update(mpl.rcParamsDefault)
# plt.rcParams.update({"font.size": FONT})

# feat_idx = list(X_used.columns).index(MAIN_FEATURE)
# x = X_used[MAIN_FEATURE].to_numpy()
# y_shap = shap_values[:, feat_idx]

# mask = np.isfinite(x) & np.isfinite(y_shap)

# if REMOVE_FLOOR:
#     floor_val, frac = infer_floor_value(x)
#     if floor_val is not None:
#         mask &= (np.round(x, 6) != np.round(floor_val, 6))
#         print(f"[{MAIN_FEATURE}] removed inferred floor={floor_val:.6f} (~{100*frac:.1f}%) for plotting")

# x = x[mask]
# y_plot = y_shap[mask]

# # Color-by feature
# c = None
# c_label = None
# if COLOR_BY is not None:
#     if COLOR_BY not in X_used.columns:
#         raise ValueError(f"COLOR_BY='{COLOR_BY}' not found in X_used columns.")
#     c = X_used.loc[X_used.index[mask], COLOR_BY].to_numpy()
#     c_label = COLOR_BY

# # -------------------------
# # NEW LAYOUT:
# # main | colorbar | right-hist
# # -------------------------
# fig = plt.figure(figsize=(FIG_W, FIG_H), dpi=300)
# gs = GridSpec(
#     2, 3,
#     width_ratios=[5.0, 0.22, 1.35],     # <— dedicated colorbar column
#     height_ratios=[1.3, 5.0],
#     wspace=0.08, hspace=0.05
# )

# ax_top   = fig.add_subplot(gs[0, 0])
# ax_main  = fig.add_subplot(gs[1, 0])
# cax      = fig.add_subplot(gs[1, 1])   # <— colorbar axis
# ax_right = fig.add_subplot(gs[1, 2])

# # Main scatter + colorbar (tight label)
# if c is None:
#     sc = ax_main.scatter(x, y_plot, s=14, alpha=0.55, edgecolor="none")
#     cax.axis("off")  # keep spacing consistent even if no color
# else:
#     sc = ax_main.scatter(x, y_plot, c=c, s=14, alpha=0.6, edgecolor="none", cmap="coolwarm")
#     cb = fig.colorbar(sc, cax=cax)
#     if c_label == "age":
#         cb.set_label("Age (years)", rotation=90, labelpad=2)  # <— closer to bar
#     else:
#         cb.set_label(f"{c_label} (feature value)", rotation=90, labelpad=2)
#     cb.ax.tick_params(pad=2)

# # Trend line
# rm = running_median_line(x, y_plot, n_bins=45, min_per_bin=35)
# if rm is not None:
#     xm, ym = rm
#     ax_main.plot(xm, ym, linewidth=2.2)

# # Robust padded limits
# xlim = robust_limits(x, pctl=X_PCTL, pad_frac=PAD_FRAC)
# ylim = robust_limits(y_plot, pctl=Y_PCTL, pad_frac=PAD_FRAC)
# if xlim:
#     ax_main.set_xlim(*xlim)
# if ylim:
#     ax_main.set_ylim(*ylim)

# # Labels
# ax_main.set_xlabel("B. longum abundance (z-score)")
# ax_main.set_ylabel("SHAP value for triglycerides")

# # Marginals
# ax_top.hist(x, bins=40, density=True)
# ax_top.set_xlim(ax_main.get_xlim())
# ax_top.axis("off")

# ax_right.hist(y_plot, bins=40, density=True, orientation="horizontal")
# ax_right.set_ylim(ax_main.get_ylim())
# ax_right.axis("off")

# # Secondary top x-axis: percentile mapping (minimal)
# xs = np.sort(x)
# ps = np.linspace(0, 100, len(xs))

# def x_to_pct(v):
#     return np.interp(v, xs, ps)

# def pct_to_x(p):
#     return np.interp(p, ps, xs)

# secax = ax_main.secondary_xaxis("top", functions=(x_to_pct, pct_to_x))
# secax.set_xlabel("Percentile")
# secax.set_xticks([10, 50, 90])

# fig.tight_layout()

# safe_pheno = sanitize_filename(PHENOTYPE)
# safe_feat = sanitize_filename(MAIN_FEATURE)
# out_png = os.path.join(OUT_DIR, f"{safe_pheno}__dependence__{safe_feat}.png")
# out_pdf = os.path.join(OUT_DIR, f"{safe_pheno}__dependence__{safe_feat}.pdf")

# fig.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
# fig.savefig(out_pdf, dpi=300, bbox_inches="tight", facecolor="white")
# plt.close(fig)

# print(f"Saved:\n  {out_png}\n  {out_pdf}")


In [ ]:
# # =========================
# # Figure 3C — SHAP dependence for ONE feature (Bifidobacterium longum)
# # Standalone: loads data, trains LGBM, computes SHAP, saves ONE dependence plot.
# #
# # Includes:
# # - Optional removal of per-feature "floor/non-existent" value (fix vertical strip)
# # - Padded percentile axis limits (avoid clipped points)
# # - Optional coloring by a covariate/interaction feature (e.g., "age" or another taxon)
# # - Running-median trend line
# # - Secondary top x-axis: cohort percentile (makes z-score easier to interpret)
# # - NO marginal histograms (clean thesis style)
# # =========================

# import os
# import re
# import pickle
# import numpy as np
# import pandas as pd
# import matplotlib as mpl
# import matplotlib.pyplot as plt
# import shap
# from lightgbm import LGBMRegressor
# from matplotlib.gridspec import GridSpec

# # -------------------------
# # USER SETTINGS
# # -------------------------
# PHENOTYPE = "bt__triglycerides"
# MAIN_FEATURE = "Bifidobacterium longum"  # must match EXACTLY a column in X

# DF_PATH = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/phenotypes_mb.pkl"
# TARGET_PHENOTYPES_PATH = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/target_phenotypes.pkl"

# OUT_DIR = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/SHAP_plots/phenotypes_predictions/thesis_fig3C"
# os.makedirs(OUT_DIR, exist_ok=True)

# BASE_FEATURES = ["age", "sex"]

# # SHAP + compute settings
# RANDOM_SEED = 0
# BACKGROUND_SIZE = 1000
# SHAP_SAMPLE_SIZE = 20000

# # Plot settings
# FIG_W, FIG_H = 6.2, 5.2
# FONT = 12

# # Color points by another feature (set None for no coloring)
# COLOR_BY = "age"   # or None

# # Axis padding / clipping prevention
# X_PCTL = (0.5, 99.5)
# Y_PCTL = (0.5, 99.5)
# PAD_FRAC = 0.12

# # Floor/non-existent value removal to fix vertical strip
# REMOVE_FLOOR = True
# DOMINANT_MIN_FRAC = 0.01
# EXTREME_MIN_Q = 0.02

# # -------------------------
# # Helpers
# # -------------------------
# def sanitize_filename(s: str) -> str:
#     return re.sub(r"[^\w\-_. ]", "_", str(s))

# def ensure_numeric_sex(series: pd.Series) -> pd.Series:
#     if pd.api.types.is_numeric_dtype(series):
#         return series
#     s = series.astype(str).str.lower().str.strip()
#     mapping = {"m": 1, "male": 1, "man": 1, "1": 1,
#                "f": 0, "female": 0, "woman": 0, "0": 0}
#     return s.map(mapping)

# def subsample_df(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
#     if len(df) <= n:
#         return df
#     return df.sample(n=n, random_state=seed)

# def median_impute(X: pd.DataFrame) -> pd.DataFrame:
#     X2 = X.copy()
#     for col in X2.columns:
#         if X2[col].isna().any():
#             X2[col] = X2[col].fillna(X2[col].median())
#     return X2

# def robust_limits(arr: np.ndarray, pctl=(0.5, 99.5), pad_frac=0.12):
#     arr = np.asarray(arr)
#     arr = arr[np.isfinite(arr)]
#     if arr.size == 0:
#         return None
#     lo, hi = np.percentile(arr, pctl)
#     if not np.isfinite(lo) or not np.isfinite(hi):
#         return None
#     if hi == lo:
#         lo -= 1.0
#         hi += 1.0
#     pad = pad_frac * (hi - lo + 1e-9)
#     return lo - pad, hi + pad

# def infer_floor_value(x: np.ndarray):
#     x = np.asarray(x)
#     x = x[np.isfinite(x)]
#     if x.size < 100:
#         return None, 0.0

#     xr = np.round(x, 6)
#     vals, counts = np.unique(xr, return_counts=True)
#     k = int(np.argmax(counts))
#     v = float(vals[k])
#     frac = counts[k] / x.size

#     if frac < DOMINANT_MIN_FRAC:
#         return None, frac

#     low_q = float(np.quantile(xr, EXTREME_MIN_Q))
#     if v <= low_q + 1e-12:
#         return v, frac

#     return None, frac

# def running_median_line(x, y, n_bins=45, min_per_bin=35):
#     x = np.asarray(x); y = np.asarray(y)
#     ok = np.isfinite(x) & np.isfinite(y)
#     x = x[ok]; y = y[ok]
#     if x.size < 200:
#         return None

#     edges = np.linspace(np.nanpercentile(x, 1), np.nanpercentile(x, 99), n_bins + 1)
#     mids, meds = [], []
#     for a, b in zip(edges[:-1], edges[1:]):
#         m = (x >= a) & (x < b)
#         if m.sum() >= min_per_bin:
#             mids.append((a + b) / 2)
#             meds.append(np.median(y[m]))
#     if len(mids) < 5:
#         return None
#     return np.array(mids), np.array(meds)

# # -------------------------
# # Load data + build X,y
# # -------------------------
# df = pd.read_pickle(DF_PATH)

# with open(TARGET_PHENOTYPES_PATH, "rb") as f:
#     target_phenotypes = pickle.load(f)

# flat_targets = sum(target_phenotypes, [])
# exclude_cols = set(flat_targets + BASE_FEATURES + ["RegistrationCode"])
# microbial_features = [c for c in df.columns if c not in exclude_cols]

# df_target = df[df[PHENOTYPE].notna()].copy()
# y = df_target[PHENOTYPE].astype(float)

# X = df_target[BASE_FEATURES + microbial_features].copy()
# X["sex"] = ensure_numeric_sex(X["sex"])

# # Drop rows with unmapped sex or missing y
# Xy = pd.concat([X, y.rename("y")], axis=1).dropna(subset=["sex", "y"])
# y = Xy["y"]
# X = Xy.drop(columns=["y"])

# # Impute
# X = median_impute(X)

# if MAIN_FEATURE not in X.columns:
#     raise ValueError(
#         f"MAIN_FEATURE='{MAIN_FEATURE}' not found in X columns. "
#         f"Example columns: {list(X.columns)[:10]} ..."
#     )

# # -------------------------
# # Train LightGBM
# # -------------------------
# n_estimators = 300 if len(y) < 3000 else 800
# learning_rate = 0.05 if len(y) < 3000 else 0.01

# model = LGBMRegressor(
#     objective="regression",
#     n_estimators=n_estimators,
#     learning_rate=learning_rate,
#     max_depth=3,
#     subsample=0.9,
#     colsample_bytree=0.8,
#     min_child_samples=5,
#     n_jobs=8,
#     random_state=RANDOM_SEED,
#     verbose=-1,
# )
# model.fit(X, y)

# # -------------------------
# # Compute SHAP on subsample
# # -------------------------
# X_used = subsample_df(X, SHAP_SAMPLE_SIZE, RANDOM_SEED)
# bg = subsample_df(X_used, min(BACKGROUND_SIZE, len(X_used)), RANDOM_SEED)

# explainer = shap.TreeExplainer(model, data=bg, feature_perturbation="interventional")
# shap_values = explainer.shap_values(X_used)
# if isinstance(shap_values, list):
#     shap_values = shap_values[0]
# shap_values = np.asarray(shap_values)

# # -------------------------
# # Plot dependence for ONE feature (NO histograms)
# # -------------------------
# mpl.rcParams.update(mpl.rcParamsDefault)
# plt.rcParams.update({"font.size": FONT})

# feat_idx = list(X_used.columns).index(MAIN_FEATURE)
# x = X_used[MAIN_FEATURE].to_numpy()
# y_shap = shap_values[:, feat_idx]

# mask = np.isfinite(x) & np.isfinite(y_shap)

# # Remove floor strip if detected
# if REMOVE_FLOOR:
#     floor_val, frac = infer_floor_value(x)
#     if floor_val is not None:
#         mask &= (np.round(x, 6) != np.round(floor_val, 6))
#         print(f"[{MAIN_FEATURE}] removed inferred floor={floor_val:.6f} (~{100*frac:.1f}%) for plotting")

# x = x[mask]
# y_plot = y_shap[mask]

# # Color-by feature
# c = None
# c_label = None
# if COLOR_BY is not None:
#     if COLOR_BY not in X_used.columns:
#         raise ValueError(f"COLOR_BY='{COLOR_BY}' not found in X_used columns.")
#     c = X_used.loc[X_used.index[mask], COLOR_BY].to_numpy()
#     c_label = COLOR_BY

# # Clean layout: main | colorbar
# fig = plt.figure(figsize=(FIG_W, FIG_H), dpi=300)
# gs = GridSpec(1, 2, width_ratios=[5.0, 0.22], wspace=0.08)

# ax_main = fig.add_subplot(gs[0, 0])
# cax = fig.add_subplot(gs[0, 1])

# # Main scatter + colorbar
# if c is None:
#     sc = ax_main.scatter(x, y_plot, s=14, alpha=0.55, edgecolor="none")
#     cax.axis("off")
# else:
#     sc = ax_main.scatter(x, y_plot, c=c, s=14, alpha=0.6, edgecolor="none", cmap="coolwarm")
#     cb = fig.colorbar(sc, cax=cax)
#     if c_label == "age":
#         cb.set_label("Age (years)", rotation=90, labelpad=2)  # close to colorbar
#     else:
#         cb.set_label(f"{c_label} (feature value)", rotation=90, labelpad=2)
#     cb.ax.tick_params(pad=2)

# # Trend line
# rm = running_median_line(x, y_plot, n_bins=45, min_per_bin=35)
# if rm is not None:
#     xm, ym = rm
#     ax_main.plot(xm, ym, linewidth=2.2)

# # Robust padded limits
# xlim = robust_limits(x, pctl=X_PCTL, pad_frac=PAD_FRAC)
# ylim = robust_limits(y_plot, pctl=Y_PCTL, pad_frac=PAD_FRAC)
# if xlim:
#     ax_main.set_xlim(*xlim)
# if ylim:
#     ax_main.set_ylim(*ylim)

# # Labels (minimal thesis style)
# ax_main.set_xlabel("B. longum abundance (z-score)")
# ax_main.set_ylabel("SHAP value for triglycerides")

# # Secondary top x-axis: percentile mapping (minimal)
# xs = np.sort(x)
# ps = np.linspace(0, 100, len(xs))

# def x_to_pct(v):
#     return np.interp(v, xs, ps)

# def pct_to_x(p):
#     return np.interp(p, ps, xs)

# secax = ax_main.secondary_xaxis("top", functions=(x_to_pct, pct_to_x))
# secax.set_xlabel("Percentile")
# secax.set_xticks([10, 50, 90])

# fig.tight_layout()

# safe_pheno = sanitize_filename(PHENOTYPE)
# safe_feat = sanitize_filename(MAIN_FEATURE)
# out_png = os.path.join(OUT_DIR, f"{safe_pheno}__dependence__{safe_feat}.png")
# out_pdf = os.path.join(OUT_DIR, f"{safe_pheno}__dependence__{safe_feat}.pdf")

# fig.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
# fig.savefig(out_pdf, dpi=300, bbox_inches="tight", facecolor="white")

# plt.show()

# plt.close(fig)

# print(f"Saved:\n  {out_png}\n  {out_pdf}")


In [ ]:
# =========================
# Figure 3C — SHAP dependence for ONE feature (Bifidobacterium longum)
# Standalone: loads data, trains LGBM, computes SHAP, saves ONE dependence plot.
#
# Includes:
# - Optional removal of per-feature "floor/non-existent" value (fix vertical strip)
# - Padded percentile axis limits (avoid clipped points)
# - Optional coloring by a covariate/interaction feature (e.g., "age" or another taxon)
# - Running-median trend line
# - Secondary top x-axis: cohort percentile (makes z-score easier to interpret)
# - TOP histogram for x distribution (aligned with percentile axis)
# =========================

import os
import re
import pickle
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import shap
from lightgbm import LGBMRegressor
from matplotlib.gridspec import GridSpec

# -------------------------
# USER SETTINGS
# -------------------------
PHENOTYPE = "bt__triglycerides"
MAIN_FEATURE = "Bifidobacterium longum"  # must match EXACTLY a column in X

DF_PATH = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/phenotypes_mb.pkl"
TARGET_PHENOTYPES_PATH = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/target_phenotypes.pkl"

OUT_DIR = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/SHAP_plots/phenotypes_predictions/thesis_fig3C"
os.makedirs(OUT_DIR, exist_ok=True)

BASE_FEATURES = ["age", "sex"]

# SHAP + compute settings
RANDOM_SEED = 0
BACKGROUND_SIZE = 1000
SHAP_SAMPLE_SIZE = 20000

# Plot settings
FIG_W, FIG_H = 6.2, 5.2
FONT = 12

# Color points by another feature (set None for no coloring)
COLOR_BY = "age"   # or None

# Axis padding / clipping prevention
X_PCTL = (0.5, 99.5)
Y_PCTL = (0.5, 99.5)
PAD_FRAC = 0.12

# Floor/non-existent value removal to fix vertical strip
REMOVE_FLOOR = True
DOMINANT_MIN_FRAC = 0.01
EXTREME_MIN_Q = 0.02

# Histogram settings (top)
TOP_HIST_BINS = 45
TOP_HIST_ALPHA = 0.9

# -------------------------
# Helpers
# -------------------------
def sanitize_filename(s: str) -> str:
    return re.sub(r"[^\w\-_. ]", "_", str(s))

def ensure_numeric_sex(series: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(series):
        return series
    s = series.astype(str).str.lower().str.strip()
    mapping = {"m": 1, "male": 1, "man": 1, "1": 1,
               "f": 0, "female": 0, "woman": 0, "0": 0}
    return s.map(mapping)

def subsample_df(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    if len(df) <= n:
        return df
    return df.sample(n=n, random_state=seed)

def median_impute(X: pd.DataFrame) -> pd.DataFrame:
    X2 = X.copy()
    for col in X2.columns:
        if X2[col].isna().any():
            X2[col] = X2[col].fillna(X2[col].median())
    return X2

def robust_limits(arr: np.ndarray, pctl=(0.5, 99.5), pad_frac=0.12):
    arr = np.asarray(arr)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return None
    lo, hi = np.percentile(arr, pctl)
    if not np.isfinite(lo) or not np.isfinite(hi):
        return None
    if hi == lo:
        lo -= 1.0
        hi += 1.0
    pad = pad_frac * (hi - lo + 1e-9)
    return lo - pad, hi + pad

def infer_floor_value(x: np.ndarray):
    x = np.asarray(x)
    x = x[np.isfinite(x)]
    if x.size < 100:
        return None, 0.0

    xr = np.round(x, 6)
    vals, counts = np.unique(xr, return_counts=True)
    k = int(np.argmax(counts))
    v = float(vals[k])
    frac = counts[k] / x.size

    if frac < DOMINANT_MIN_FRAC:
        return None, frac

    low_q = float(np.quantile(xr, EXTREME_MIN_Q))
    if v <= low_q + 1e-12:
        return v, frac

    return None, frac

def running_median_line(x, y, n_bins=45, min_per_bin=35):
    x = np.asarray(x); y = np.asarray(y)
    ok = np.isfinite(x) & np.isfinite(y)
    x = x[ok]; y = y[ok]
    if x.size < 200:
        return None

    edges = np.linspace(np.nanpercentile(x, 1), np.nanpercentile(x, 99), n_bins + 1)
    mids, meds = [], []
    for a, b in zip(edges[:-1], edges[1:]):
        m = (x >= a) & (x < b)
        if m.sum() >= min_per_bin:
            mids.append((a + b) / 2)
            meds.append(np.median(y[m]))
    if len(mids) < 5:
        return None
    return np.array(mids), np.array(meds)

# -------------------------
# Load data + build X,y
# -------------------------
df = pd.read_pickle(DF_PATH)

with open(TARGET_PHENOTYPES_PATH, "rb") as f:
    target_phenotypes = pickle.load(f)

flat_targets = sum(target_phenotypes, [])
exclude_cols = set(flat_targets + BASE_FEATURES + ["RegistrationCode"])
microbial_features = [c for c in df.columns if c not in exclude_cols]

df_target = df[df[PHENOTYPE].notna()].copy()
y = df_target[PHENOTYPE].astype(float)

X = df_target[BASE_FEATURES + microbial_features].copy()
X["sex"] = ensure_numeric_sex(X["sex"])

# Drop rows with unmapped sex or missing y
Xy = pd.concat([X, y.rename("y")], axis=1).dropna(subset=["sex", "y"])
y = Xy["y"]
X = Xy.drop(columns=["y"])

# Impute
X = median_impute(X)

if MAIN_FEATURE not in X.columns:
    raise ValueError(
        f"MAIN_FEATURE='{MAIN_FEATURE}' not found in X columns. "
        f"Example columns: {list(X.columns)[:10]} ..."
    )

# -------------------------
# Train LightGBM
# -------------------------
n_estimators = 300 if len(y) < 3000 else 800
learning_rate = 0.05 if len(y) < 3000 else 0.01

model = LGBMRegressor(
    objective="regression",
    n_estimators=n_estimators,
    learning_rate=learning_rate,
    max_depth=3,
    subsample=0.9,
    colsample_bytree=0.8,
    min_child_samples=5,
    n_jobs=8,
    random_state=RANDOM_SEED,
    verbose=-1,
)
model.fit(X, y)

# -------------------------
# Compute SHAP on subsample
# -------------------------
X_used = subsample_df(X, SHAP_SAMPLE_SIZE, RANDOM_SEED)
bg = subsample_df(X_used, min(BACKGROUND_SIZE, len(X_used)), RANDOM_SEED)

explainer = shap.TreeExplainer(model, data=bg, feature_perturbation="interventional")
shap_values = explainer.shap_values(X_used)
if isinstance(shap_values, list):
    shap_values = shap_values[0]
shap_values = np.asarray(shap_values)

# -------------------------
# Plot dependence for ONE feature (TOP histogram + colorbar)
# -------------------------
mpl.rcParams.update(mpl.rcParamsDefault)
plt.rcParams.update({"font.size": FONT})

feat_idx = list(X_used.columns).index(MAIN_FEATURE)
x = X_used[MAIN_FEATURE].to_numpy()
y_shap = shap_values[:, feat_idx]

mask = np.isfinite(x) & np.isfinite(y_shap)

# Remove floor strip if detected
if REMOVE_FLOOR:
    floor_val, frac = infer_floor_value(x)
    if floor_val is not None:
        mask &= (np.round(x, 6) != np.round(floor_val, 6))
        print(f"[{MAIN_FEATURE}] removed inferred floor={floor_val:.6f} (~{100*frac:.1f}%) for plotting")

x = x[mask]
y_plot = y_shap[mask]

# Color-by feature
c = None
c_label = None
if COLOR_BY is not None:
    if COLOR_BY not in X_used.columns:
        raise ValueError(f"COLOR_BY='{COLOR_BY}' not found in X_used columns.")
    c = X_used.loc[X_used.index[mask], COLOR_BY].to_numpy()
    c_label = COLOR_BY

# Layout: top hist over main, and colorbar column on the right
fig = plt.figure(figsize=(FIG_W, FIG_H), dpi=300)
gs = GridSpec(
    2, 2,
    width_ratios=[5.0, 0.22],
    height_ratios=[1.1, 5.0],
    wspace=0.08, hspace=0.05
)

ax_top  = fig.add_subplot(gs[0, 0])
ax_main = fig.add_subplot(gs[1, 0])
cax     = fig.add_subplot(gs[1, 1])

# Main scatter + colorbar
if c is None:
    sc = ax_main.scatter(x, y_plot, s=14, alpha=0.55, edgecolor="none")
    cax.axis("off")
else:
    sc = ax_main.scatter(x, y_plot, c=c, s=14, alpha=0.6, edgecolor="none", cmap="coolwarm")
    cb = fig.colorbar(sc, cax=cax)
    if c_label == "age":
        cb.set_label("Age (years)", rotation=90, labelpad=2)  # close to colorbar
    else:
        cb.set_label(f"{c_label} (feature value)", rotation=90, labelpad=2)
    cb.ax.tick_params(pad=2)

# Trend line
rm = running_median_line(x, y_plot, n_bins=45, min_per_bin=35)
if rm is not None:
    xm, ym = rm
    ax_main.plot(xm, ym, linewidth=2.2)

# Robust padded limits
xlim = robust_limits(x, pctl=X_PCTL, pad_frac=PAD_FRAC)
ylim = robust_limits(y_plot, pctl=Y_PCTL, pad_frac=PAD_FRAC)
if xlim:
    ax_main.set_xlim(*xlim)
if ylim:
    ax_main.set_ylim(*ylim)

# Labels
ax_main.set_xlabel("B. longum abundance (z-score)")
ax_main.set_ylabel("SHAP value for triglycerides (mg/dL)")

# Secondary top x-axis: percentile mapping (on the MAIN axis)
xs = np.sort(x)
ps = np.linspace(0, 100, len(xs))

def x_to_pct(v):
    return np.interp(v, xs, ps)

def pct_to_x(p):
    return np.interp(p, ps, xs)

secax = ax_main.secondary_xaxis("top", functions=(x_to_pct, pct_to_x))
secax.set_xlabel("Percentile")

secax.set_xticks([10, 50, 90])

# Top histogram (aligned to main x-limits)
ax_top.hist(x, bins=TOP_HIST_BINS, density=True, alpha=TOP_HIST_ALPHA)
ax_top.set_xlim(ax_main.get_xlim())
ax_top.axis("off")

# Keep the upper histogram but avoid it covering the percentile axis label
# (Percentile axis is on ax_main; histogram is a separate axes above)
# You can fine-tune spacing via height_ratios / hspace if needed.

fig.tight_layout()

safe_pheno = sanitize_filename(PHENOTYPE)
safe_feat = sanitize_filename(MAIN_FEATURE)
out_png = os.path.join(OUT_DIR, f"{safe_pheno}__dependence__{safe_feat}.png")
out_pdf = os.path.join(OUT_DIR, f"{safe_pheno}__dependence__{safe_feat}.pdf")

fig.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
fig.savefig(out_pdf, dpi=300, bbox_inches="tight", facecolor="white")

plt.show()

plt.close(fig)

print(f"Saved:\n  {out_png}\n  {out_pdf}")


In [ ]:
# =========================
# Figure 3C — SHAP dependence for ONE feature (Bifidobacterium longum)
# Standalone: loads data, trains LGBM, computes SHAP, saves ONE dependence plot.
# =========================

import os
import re
import pickle
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import shap
from lightgbm import LGBMRegressor
from matplotlib.gridspec import GridSpec

# -------------------------
# USER SETTINGS
# -------------------------
PHENOTYPE = "bt__triglycerides"
MAIN_FEATURE = "Bifidobacterium longum"  # must match EXACTLY a column in X

DF_PATH = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/phenotypes_mb.pkl"
TARGET_PHENOTYPES_PATH = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/target_phenotypes.pkl"

# --- NEW: raw age source ---
DIET_MB_PATH = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/segal_species/diet_mb.pkl"

OUT_DIR = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/SHAP_plots/phenotypes_predictions/thesis_fig3C"
os.makedirs(OUT_DIR, exist_ok=True)

BASE_FEATURES = ["age", "sex"]

# SHAP + compute settings
RANDOM_SEED = 0
BACKGROUND_SIZE = 1000
SHAP_SAMPLE_SIZE = 20000

# Plot settings
FIG_W, FIG_H = 6.2, 5.2
FONT = 12

# Color points by another feature (set None for no coloring)
COLOR_BY = "age"   # or None

# Axis padding / clipping prevention
X_PCTL = (0.5, 99.5)
Y_PCTL = (0.5, 99.5)
PAD_FRAC = 0.12

# Floor/non-existent value removal to fix vertical strip
REMOVE_FLOOR = True
DOMINANT_MIN_FRAC = 0.01
EXTREME_MIN_Q = 0.02

# Histogram settings (top)
TOP_HIST_BINS = 45
TOP_HIST_ALPHA = 0.9

# Percentile label placement (relative to histogram)
PCTL_LABEL_POS = "above"   # "above" or "below"
PCTL_LABEL_DY = 0.012

# -------------------------
# Helpers
# -------------------------
def sanitize_filename(s: str) -> str:
    return re.sub(r"[^\w\-_. ]", "_", str(s))

def ensure_numeric_sex(series: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(series):
        return series
    s = series.astype(str).str.lower().str.strip()
    mapping = {"m": 1, "male": 1, "man": 1, "1": 1,
               "f": 0, "female": 0, "woman": 0, "0": 0}
    return s.map(mapping)

def subsample_df(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    if len(df) <= n:
        return df
    return df.sample(n=n, random_state=seed)

def median_impute(X: pd.DataFrame) -> pd.DataFrame:
    X2 = X.copy()
    for col in X2.columns:
        if X2[col].isna().any():
            X2[col] = X2[col].fillna(X2[col].median())
    return X2

def robust_limits(arr: np.ndarray, pctl=(0.5, 99.5), pad_frac=0.12):
    arr = np.asarray(arr)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return None
    lo, hi = np.percentile(arr, pctl)
    if not np.isfinite(lo) or not np.isfinite(hi):
        return None
    if hi == lo:
        lo -= 1.0
        hi += 1.0
    pad = pad_frac * (hi - lo + 1e-9)
    return lo - pad, hi + pad

def infer_floor_value(x: np.ndarray):
    x = np.asarray(x)
    x = x[np.isfinite(x)]
    if x.size < 100:
        return None, 0.0

    xr = np.round(x, 6)
    vals, counts = np.unique(xr, return_counts=True)
    k = int(np.argmax(counts))
    v = float(vals[k])
    frac = counts[k] / x.size

    if frac < DOMINANT_MIN_FRAC:
        return None, frac

    low_q = float(np.quantile(xr, EXTREME_MIN_Q))
    if v <= low_q + 1e-12:
        return v, frac

    return None, frac

def running_median_line(x, y, n_bins=45, min_per_bin=35):
    x = np.asarray(x); y = np.asarray(y)
    ok = np.isfinite(x) & np.isfinite(y)
    x = x[ok]; y = y[ok]
    if x.size < 200:
        return None

    edges = np.linspace(np.nanpercentile(x, 1), np.nanpercentile(x, 99), n_bins + 1)
    mids, meds = [], []
    for a, b in zip(edges[:-1], edges[1:]):
        m = (x >= a) & (x < b)
        if m.sum() >= min_per_bin:
            mids.append((a + b) / 2)
            meds.append(np.median(y[m]))
    if len(mids) < 5:
        return None
    return np.array(mids), np.array(meds)

# -------------------------
# Load data
# -------------------------
df = pd.read_pickle(DF_PATH)

with open(TARGET_PHENOTYPES_PATH, "rb") as f:
    target_phenotypes = pickle.load(f)

# --- NEW: load diet_mb + inject raw ages (years) using RegistrationCode ---
diet_mb = pd.read_pickle(DIET_MB_PATH)

# Build RegistrationCode -> age(years) mapping robustly
if "RegistrationCode" in diet_mb.columns:
    age_by_rc = diet_mb.copy()
    age_by_rc["age"] = pd.to_numeric(age_by_rc["age"], errors="coerce")
    age_by_rc = age_by_rc.groupby("RegistrationCode")["age"].median()
else:
    # assume index is RegistrationCode
    if "age" not in diet_mb.columns:
        raise ValueError("diet_mb does not have an 'age' column.")
    age_by_rc = pd.to_numeric(diet_mb["age"], errors="coerce")
    if diet_mb.index.has_duplicates:
        age_by_rc = pd.to_numeric(diet_mb.groupby(level=0)["age"].median(), errors="coerce")

# -------------------------
# Build X,y
# -------------------------
flat_targets = sum(target_phenotypes, [])
exclude_cols = set(flat_targets + BASE_FEATURES + ["RegistrationCode"])
microbial_features = [c for c in df.columns if c not in exclude_cols]

df_target = df[df[PHENOTYPE].notna()].copy()
y = df_target[PHENOTYPE].astype(float)

# --- NEW: overwrite df_target["age"] with RAW age (years) from diet_mb ---
if "RegistrationCode" not in df_target.columns:
    raise ValueError("df_target is missing 'RegistrationCode' column needed for merging raw age.")

df_target["age"] = pd.to_numeric(df_target["RegistrationCode"].map(age_by_rc), errors="coerce")

miss_frac = df_target["age"].isna().mean()
if miss_frac > 0:
    print(f"[Warning] raw age missing for {miss_frac*100:.1f}% of rows after mapping from diet_mb. "
          f"Will median-impute for modeling/plotting.")

X = df_target[BASE_FEATURES + microbial_features].copy()
X["sex"] = ensure_numeric_sex(X["sex"])

# Drop rows with unmapped sex or missing y
Xy = pd.concat([X, y.rename("y")], axis=1).dropna(subset=["sex", "y"])
y = Xy["y"]
X = Xy.drop(columns=["y"])

# Impute (includes any missing age after mapping)
X = median_impute(X)

if MAIN_FEATURE not in X.columns:
    raise ValueError(
        f"MAIN_FEATURE='{MAIN_FEATURE}' not found in X columns. "
        f"Example columns: {list(X.columns)[:10]} ..."
    )

# -------------------------
# Train LightGBM
# -------------------------
n_estimators = 300 if len(y) < 3000 else 800
learning_rate = 0.05 if len(y) < 3000 else 0.01

model = LGBMRegressor(
    objective="regression",
    n_estimators=n_estimators,
    learning_rate=learning_rate,
    max_depth=3,
    subsample=0.9,
    colsample_bytree=0.8,
    min_child_samples=5,
    n_jobs=8,
    random_state=RANDOM_SEED,
    verbose=-1,
)
model.fit(X, y)

# -------------------------
# Compute SHAP on subsample
# -------------------------
X_used = subsample_df(X, SHAP_SAMPLE_SIZE, RANDOM_SEED)
bg = subsample_df(X_used, min(BACKGROUND_SIZE, len(X_used)), RANDOM_SEED)

explainer = shap.TreeExplainer(model, data=bg, feature_perturbation="interventional")
shap_values = explainer.shap_values(X_used)
if isinstance(shap_values, list):
    shap_values = shap_values[0]
shap_values = np.asarray(shap_values)

# -------------------------
# Plot dependence for ONE feature (TOP histogram + colorbar)
# -------------------------
mpl.rcParams.update(mpl.rcParamsDefault)
plt.rcParams.update({"font.size": FONT})

feat_idx = list(X_used.columns).index(MAIN_FEATURE)
x = X_used[MAIN_FEATURE].to_numpy()
y_shap = shap_values[:, feat_idx]

mask = np.isfinite(x) & np.isfinite(y_shap)

if REMOVE_FLOOR:
    floor_val, frac = infer_floor_value(x)
    if floor_val is not None:
        mask &= (np.round(x, 6) != np.round(floor_val, 6))
        print(f"[{MAIN_FEATURE}] removed inferred floor={floor_val:.6f} (~{100*frac:.1f}%) for plotting")

x = x[mask]
y_plot = y_shap[mask]

# Color-by feature
c = None
c_label = None
if COLOR_BY is not None:
    if COLOR_BY not in X_used.columns:
        raise ValueError(f"COLOR_BY='{COLOR_BY}' not found in X_used columns.")
    c = X_used.loc[X_used.index[mask], COLOR_BY].to_numpy()
    c_label = COLOR_BY

fig = plt.figure(figsize=(FIG_W, FIG_H), dpi=300)
gs = GridSpec(
    2, 2,
    width_ratios=[5.0, 0.22],
    height_ratios=[1.1, 5.0],
    wspace=0.08, hspace=0.05
)

ax_top  = fig.add_subplot(gs[0, 0])
ax_main = fig.add_subplot(gs[1, 0])
cax     = fig.add_subplot(gs[1, 1])

# Main scatter + colorbar
if c is None:
    sc = ax_main.scatter(x, y_plot, s=14, alpha=0.55, edgecolor="none")
    cax.axis("off")
else:
    sc = ax_main.scatter(x, y_plot, c=c, s=14, alpha=0.6, edgecolor="none", cmap="coolwarm")
    cb = fig.colorbar(sc, cax=cax)
    if c_label == "age":
        cb.set_label("Age (years)", rotation=90, labelpad=10)
    else:
        cb.set_label(f"{c_label} (feature value)", rotation=90, labelpad=2)
    cb.ax.tick_params(pad=2)

# Trend line
rm = running_median_line(x, y_plot, n_bins=45, min_per_bin=35)
if rm is not None:
    xm, ym = rm
    ax_main.plot(xm, ym, linewidth=2.2)

# Robust padded limits
xlim = robust_limits(x, pctl=X_PCTL, pad_frac=PAD_FRAC)
ylim = robust_limits(y_plot, pctl=Y_PCTL, pad_frac=PAD_FRAC)
if xlim:
    ax_main.set_xlim(*xlim)
if ylim:
    ax_main.set_ylim(*ylim)

# Labels
ax_main.set_xlabel(r"$\it{B.\,longum}$ abundance (z-score)")
ax_main.set_ylabel("SHAP value for triglycerides (mg/dL)", labelpad=8)

# Secondary top x-axis: percentile mapping
xs = np.sort(x)
ps = np.linspace(0, 100, len(xs))

def x_to_pct(v):
    return np.interp(v, xs, ps)

def pct_to_x(p):
    return np.interp(p, ps, xs)

secax = ax_main.secondary_xaxis("top", functions=(x_to_pct, pct_to_x))
secax.set_xlabel("")                 # place manually
secax.set_xticks([10, 50, 90])

# Top histogram
ax_top.hist(x, bins=TOP_HIST_BINS, density=True, alpha=TOP_HIST_ALPHA)
ax_top.set_xlim(ax_main.get_xlim())
ax_top.axis("off")
ax_top.patch.set_alpha(0.0)

# Layout then place "Percentile"
fig.tight_layout()

top_bbox = ax_top.get_position()
x_center = 0.5 * (top_bbox.x0 + top_bbox.x1)

if str(PCTL_LABEL_POS).lower().startswith("above"):
    y_text = top_bbox.y1 + PCTL_LABEL_DY
    va = "bottom"
else:
    y_text = top_bbox.y0 - PCTL_LABEL_DY
    va = "top"

fig.text(x_center, y_text, "Percentile", ha="center", va=va, fontsize=FONT)

safe_pheno = sanitize_filename(PHENOTYPE)
safe_feat = sanitize_filename(MAIN_FEATURE)
out_png = os.path.join(OUT_DIR, f"{safe_pheno}__dependence__{safe_feat}.png")
out_pdf = os.path.join(OUT_DIR, f"{safe_pheno}__dependence__{safe_feat}.pdf")

fig.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
fig.savefig(out_pdf, dpi=300, bbox_inches="tight", facecolor="white")

plt.show()
plt.close(fig)

print(f"Saved:\n  {out_png}\n  {out_pdf}")


In [ ]:
# =========================
# Figure 3C — SHAP dependence for ONE feature (Bifidobacterium longum)
# Standalone: loads data, trains LGBM, computes SHAP, saves ONE dependence plot.
# =========================

import os
import re
import pickle
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import shap
from lightgbm import LGBMRegressor
from matplotlib.gridspec import GridSpec

# -------------------------
# USER SETTINGS
# -------------------------
PHENOTYPE = "triglyceride_to_hdl_ratio"
MAIN_FEATURE = "Bifidobacterium longum"  # must match EXACTLY a column in X

DF_PATH = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/phenotypes_mb.pkl"
TARGET_PHENOTYPES_PATH = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/target_phenotypes.pkl"

# --- NEW: raw age source ---
DIET_MB_PATH = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/segal_species/diet_mb.pkl"

OUT_DIR = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/SHAP_plots/phenotypes_predictions/thesis_fig3C"
os.makedirs(OUT_DIR, exist_ok=True)

BASE_FEATURES = ["age", "sex"]

# SHAP + compute settings
RANDOM_SEED = 0
BACKGROUND_SIZE = 1000
SHAP_SAMPLE_SIZE = 20000

# Plot settings
FIG_W, FIG_H = 6.2, 5.2
FONT = 12

# Color points by another feature (set None for no coloring)
COLOR_BY = "age"   # or None

# Axis padding / clipping prevention
X_PCTL = (0.5, 99.5)
Y_PCTL = (0.5, 99.5)
PAD_FRAC = 0.12

# Floor/non-existent value removal to fix vertical strip
REMOVE_FLOOR = True
DOMINANT_MIN_FRAC = 0.01
EXTREME_MIN_Q = 0.02

# Histogram settings (top)
TOP_HIST_BINS = 45
TOP_HIST_ALPHA = 0.9

# Percentile label placement (relative to histogram)
PCTL_LABEL_POS = "above"   # "above" or "below"
PCTL_LABEL_DY = 0.012

# -------------------------
# Helpers
# -------------------------
def sanitize_filename(s: str) -> str:
    return re.sub(r"[^\w\-_. ]", "_", str(s))

def ensure_numeric_sex(series: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(series):
        return series
    s = series.astype(str).str.lower().str.strip()
    mapping = {"m": 1, "male": 1, "man": 1, "1": 1,
               "f": 0, "female": 0, "woman": 0, "0": 0}
    return s.map(mapping)

def subsample_df(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    if len(df) <= n:
        return df
    return df.sample(n=n, random_state=seed)

def median_impute(X: pd.DataFrame) -> pd.DataFrame:
    X2 = X.copy()
    for col in X2.columns:
        if X2[col].isna().any():
            X2[col] = X2[col].fillna(X2[col].median())
    return X2

def robust_limits(arr: np.ndarray, pctl=(0.5, 99.5), pad_frac=0.12):
    arr = np.asarray(arr)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return None
    lo, hi = np.percentile(arr, pctl)
    if not np.isfinite(lo) or not np.isfinite(hi):
        return None
    if hi == lo:
        lo -= 1.0
        hi += 1.0
    pad = pad_frac * (hi - lo + 1e-9)
    return lo - pad, hi + pad

def infer_floor_value(x: np.ndarray):
    x = np.asarray(x)
    x = x[np.isfinite(x)]
    if x.size < 100:
        return None, 0.0

    xr = np.round(x, 6)
    vals, counts = np.unique(xr, return_counts=True)
    k = int(np.argmax(counts))
    v = float(vals[k])
    frac = counts[k] / x.size

    if frac < DOMINANT_MIN_FRAC:
        return None, frac

    low_q = float(np.quantile(xr, EXTREME_MIN_Q))
    if v <= low_q + 1e-12:
        return v, frac

    return None, frac

def running_median_line(x, y, n_bins=45, min_per_bin=35):
    x = np.asarray(x); y = np.asarray(y)
    ok = np.isfinite(x) & np.isfinite(y)
    x = x[ok]; y = y[ok]
    if x.size < 200:
        return None

    edges = np.linspace(np.nanpercentile(x, 1), np.nanpercentile(x, 99), n_bins + 1)
    mids, meds = [], []
    for a, b in zip(edges[:-1], edges[1:]):
        m = (x >= a) & (x < b)
        if m.sum() >= min_per_bin:
            mids.append((a + b) / 2)
            meds.append(np.median(y[m]))
    if len(mids) < 5:
        return None
    return np.array(mids), np.array(meds)

# -------------------------
# Load data
# -------------------------
df = pd.read_pickle(DF_PATH)

with open(TARGET_PHENOTYPES_PATH, "rb") as f:
    target_phenotypes = pickle.load(f)

# --- NEW: load diet_mb + inject raw ages (years) using RegistrationCode ---
diet_mb = pd.read_pickle(DIET_MB_PATH)

# Build RegistrationCode -> age(years) mapping robustly
if "RegistrationCode" in diet_mb.columns:
    age_by_rc = diet_mb.copy()
    age_by_rc["age"] = pd.to_numeric(age_by_rc["age"], errors="coerce")
    age_by_rc = age_by_rc.groupby("RegistrationCode")["age"].median()
else:
    # assume index is RegistrationCode
    if "age" not in diet_mb.columns:
        raise ValueError("diet_mb does not have an 'age' column.")
    age_by_rc = pd.to_numeric(diet_mb["age"], errors="coerce")
    if diet_mb.index.has_duplicates:
        age_by_rc = pd.to_numeric(diet_mb.groupby(level=0)["age"].median(), errors="coerce")

# -------------------------
# Build X,y
# -------------------------
flat_targets = sum(target_phenotypes, [])
exclude_cols = set(flat_targets + BASE_FEATURES + ["RegistrationCode"])
microbial_features = [c for c in df.columns if c not in exclude_cols]

df_target = df[df[PHENOTYPE].notna()].copy()
y = df_target[PHENOTYPE].astype(float)

# --- NEW: overwrite df_target["age"] with RAW age (years) from diet_mb ---
if "RegistrationCode" not in df_target.columns:
    raise ValueError("df_target is missing 'RegistrationCode' column needed for merging raw age.")

df_target["age"] = pd.to_numeric(df_target["RegistrationCode"].map(age_by_rc), errors="coerce")

miss_frac = df_target["age"].isna().mean()
if miss_frac > 0:
    print(f"[Warning] raw age missing for {miss_frac*100:.1f}% of rows after mapping from diet_mb. "
          f"Will median-impute for modeling/plotting.")

X = df_target[BASE_FEATURES + microbial_features].copy()
X["sex"] = ensure_numeric_sex(X["sex"])

# Drop rows with unmapped sex or missing y
Xy = pd.concat([X, y.rename("y")], axis=1).dropna(subset=["sex", "y"])
y = Xy["y"]
X = Xy.drop(columns=["y"])

# Impute (includes any missing age after mapping)
X = median_impute(X)

if MAIN_FEATURE not in X.columns:
    raise ValueError(
        f"MAIN_FEATURE='{MAIN_FEATURE}' not found in X columns. "
        f"Example columns: {list(X.columns)[:10]} ..."
    )

# -------------------------
# Train LightGBM
# -------------------------
n_estimators = 300 if len(y) < 3000 else 800
learning_rate = 0.05 if len(y) < 3000 else 0.01

model = LGBMRegressor(
    objective="regression",
    n_estimators=n_estimators,
    learning_rate=learning_rate,
    max_depth=3,
    subsample=0.9,
    colsample_bytree=0.8,
    min_child_samples=5,
    n_jobs=8,
    random_state=RANDOM_SEED,
    verbose=-1,
)
model.fit(X, y)

# -------------------------
# Compute SHAP on subsample
# -------------------------
X_used = subsample_df(X, SHAP_SAMPLE_SIZE, RANDOM_SEED)
bg = subsample_df(X_used, min(BACKGROUND_SIZE, len(X_used)), RANDOM_SEED)

explainer = shap.TreeExplainer(model, data=bg, feature_perturbation="interventional")
shap_values = explainer.shap_values(X_used)
if isinstance(shap_values, list):
    shap_values = shap_values[0]
shap_values = np.asarray(shap_values)

# -------------------------
# Plot dependence for ONE feature (TOP histogram + colorbar)
# -------------------------
mpl.rcParams.update(mpl.rcParamsDefault)
plt.rcParams.update({"font.size": FONT})

feat_idx = list(X_used.columns).index(MAIN_FEATURE)
x = X_used[MAIN_FEATURE].to_numpy()
y_shap = shap_values[:, feat_idx]

mask = np.isfinite(x) & np.isfinite(y_shap)

if REMOVE_FLOOR:
    floor_val, frac = infer_floor_value(x)
    if floor_val is not None:
        mask &= (np.round(x, 6) != np.round(floor_val, 6))
        print(f"[{MAIN_FEATURE}] removed inferred floor={floor_val:.6f} (~{100*frac:.1f}%) for plotting")

x = x[mask]
y_plot = y_shap[mask]

# Color-by feature
c = None
c_label = None
if COLOR_BY is not None:
    if COLOR_BY not in X_used.columns:
        raise ValueError(f"COLOR_BY='{COLOR_BY}' not found in X_used columns.")
    c = X_used.loc[X_used.index[mask], COLOR_BY].to_numpy()
    c_label = COLOR_BY

fig = plt.figure(figsize=(FIG_W, FIG_H), dpi=300)
gs = GridSpec(
    2, 2,
    width_ratios=[5.0, 0.22],
    height_ratios=[1.1, 5.0],
    wspace=0.08, hspace=0.05
)

ax_top  = fig.add_subplot(gs[0, 0])
ax_main = fig.add_subplot(gs[1, 0])
cax     = fig.add_subplot(gs[1, 1])

# Main scatter + colorbar
if c is None:
    sc = ax_main.scatter(x, y_plot, s=14, alpha=0.55, edgecolor="none")
    cax.axis("off")
else:
    sc = ax_main.scatter(x, y_plot, c=c, s=14, alpha=0.6, edgecolor="none", cmap="coolwarm")
    cb = fig.colorbar(sc, cax=cax)
    if c_label == "age":
        cb.set_label("Age (years)", rotation=90, labelpad=10)
    else:
        cb.set_label(f"{c_label} (feature value)", rotation=90, labelpad=2)
    cb.ax.tick_params(pad=2)

# Trend line
rm = running_median_line(x, y_plot, n_bins=45, min_per_bin=35)
if rm is not None:
    xm, ym = rm
    ax_main.plot(xm, ym, linewidth=2.2)

# Robust padded limits
xlim = robust_limits(x, pctl=X_PCTL, pad_frac=PAD_FRAC)
ylim = robust_limits(y_plot, pctl=Y_PCTL, pad_frac=PAD_FRAC)
if xlim:
    ax_main.set_xlim(*xlim)
if ylim:
    ax_main.set_ylim(*ylim)

# Labels
ax_main.set_xlabel("B. longum abundance (z-score)")
ax_main.set_ylabel("SHAP value for triglyceride/HDL ratio", labelpad=8)

# Secondary top x-axis: percentile mapping
xs = np.sort(x)
ps = np.linspace(0, 100, len(xs))

def x_to_pct(v):
    return np.interp(v, xs, ps)

def pct_to_x(p):
    return np.interp(p, ps, xs)

secax = ax_main.secondary_xaxis("top", functions=(x_to_pct, pct_to_x))
secax.set_xlabel("")                 # place manually
secax.set_xticks([10, 50, 90])

# Top histogram
ax_top.hist(x, bins=TOP_HIST_BINS, density=True, alpha=TOP_HIST_ALPHA)
ax_top.set_xlim(ax_main.get_xlim())
ax_top.axis("off")
ax_top.patch.set_alpha(0.0)

# Layout then place "Percentile"
fig.tight_layout()

top_bbox = ax_top.get_position()
x_center = 0.5 * (top_bbox.x0 + top_bbox.x1)

if str(PCTL_LABEL_POS).lower().startswith("above"):
    y_text = top_bbox.y1 + PCTL_LABEL_DY
    va = "bottom"
else:
    y_text = top_bbox.y0 - PCTL_LABEL_DY
    va = "top"

fig.text(x_center, y_text, "Percentile", ha="center", va=va, fontsize=FONT)

safe_pheno = sanitize_filename(PHENOTYPE)
safe_feat = sanitize_filename(MAIN_FEATURE)
out_png = os.path.join(OUT_DIR, f"{safe_pheno}__dependence__{safe_feat}.png")
out_pdf = os.path.join(OUT_DIR, f"{safe_pheno}__dependence__{safe_feat}.pdf")

fig.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
fig.savefig(out_pdf, dpi=300, bbox_inches="tight", facecolor="white")

plt.show()
plt.close(fig)

print(f"Saved:\n  {out_png}\n  {out_pdf}")


In [ ]:
# =========================
# Figure 3C — SHAP dependence for ONE feature (Bifidobacterium longum)
# Standalone: loads data, trains LGBM, computes SHAP, saves ONE dependence plot.
# =========================

import os
import re
import pickle
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import shap
from lightgbm import LGBMRegressor
from matplotlib.gridspec import GridSpec

# -------------------------
# USER SETTINGS
# -------------------------
PHENOTYPE = "total_scan_vat_mass"
MAIN_FEATURE = "Bifidobacterium longum"  # must match EXACTLY a column in X

DF_PATH = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/phenotypes_mb.pkl"
TARGET_PHENOTYPES_PATH = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/target_phenotypes.pkl"

# --- NEW: raw age source ---
DIET_MB_PATH = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/segal_species/diet_mb.pkl"

OUT_DIR = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/SHAP_plots/phenotypes_predictions/thesis_fig3C"
os.makedirs(OUT_DIR, exist_ok=True)

BASE_FEATURES = ["age", "sex"]

# SHAP + compute settings
RANDOM_SEED = 0
BACKGROUND_SIZE = 1000
SHAP_SAMPLE_SIZE = 20000

# Plot settings
FIG_W, FIG_H = 6.2, 5.2
FONT = 12

# Color points by another feature (set None for no coloring)
COLOR_BY = "age"   # or None

# Axis padding / clipping prevention
X_PCTL = (0.5, 99.5)
Y_PCTL = (0.5, 99.5)
PAD_FRAC = 0.12

# Floor/non-existent value removal to fix vertical strip
REMOVE_FLOOR = True
DOMINANT_MIN_FRAC = 0.01
EXTREME_MIN_Q = 0.02

# Histogram settings (top)
TOP_HIST_BINS = 45
TOP_HIST_ALPHA = 0.9

# Percentile label placement (relative to histogram)
PCTL_LABEL_POS = "above"   # "above" or "below"
PCTL_LABEL_DY = 0.012

# -------------------------
# Helpers
# -------------------------
def sanitize_filename(s: str) -> str:
    return re.sub(r"[^\w\-_. ]", "_", str(s))

def ensure_numeric_sex(series: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(series):
        return series
    s = series.astype(str).str.lower().str.strip()
    mapping = {"m": 1, "male": 1, "man": 1, "1": 1,
               "f": 0, "female": 0, "woman": 0, "0": 0}
    return s.map(mapping)

def subsample_df(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    if len(df) <= n:
        return df
    return df.sample(n=n, random_state=seed)

def median_impute(X: pd.DataFrame) -> pd.DataFrame:
    X2 = X.copy()
    for col in X2.columns:
        if X2[col].isna().any():
            X2[col] = X2[col].fillna(X2[col].median())
    return X2

def robust_limits(arr: np.ndarray, pctl=(0.5, 99.5), pad_frac=0.12):
    arr = np.asarray(arr)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return None
    lo, hi = np.percentile(arr, pctl)
    if not np.isfinite(lo) or not np.isfinite(hi):
        return None
    if hi == lo:
        lo -= 1.0
        hi += 1.0
    pad = pad_frac * (hi - lo + 1e-9)
    return lo - pad, hi + pad

def infer_floor_value(x: np.ndarray):
    x = np.asarray(x)
    x = x[np.isfinite(x)]
    if x.size < 100:
        return None, 0.0

    xr = np.round(x, 6)
    vals, counts = np.unique(xr, return_counts=True)
    k = int(np.argmax(counts))
    v = float(vals[k])
    frac = counts[k] / x.size

    if frac < DOMINANT_MIN_FRAC:
        return None, frac

    low_q = float(np.quantile(xr, EXTREME_MIN_Q))
    if v <= low_q + 1e-12:
        return v, frac

    return None, frac

def running_median_line(x, y, n_bins=45, min_per_bin=35):
    x = np.asarray(x); y = np.asarray(y)
    ok = np.isfinite(x) & np.isfinite(y)
    x = x[ok]; y = y[ok]
    if x.size < 200:
        return None

    edges = np.linspace(np.nanpercentile(x, 1), np.nanpercentile(x, 99), n_bins + 1)
    mids, meds = [], []
    for a, b in zip(edges[:-1], edges[1:]):
        m = (x >= a) & (x < b)
        if m.sum() >= min_per_bin:
            mids.append((a + b) / 2)
            meds.append(np.median(y[m]))
    if len(mids) < 5:
        return None
    return np.array(mids), np.array(meds)

# -------------------------
# Load data
# -------------------------
df = pd.read_pickle(DF_PATH)

with open(TARGET_PHENOTYPES_PATH, "rb") as f:
    target_phenotypes = pickle.load(f)

# --- NEW: load diet_mb + inject raw ages (years) using RegistrationCode ---
diet_mb = pd.read_pickle(DIET_MB_PATH)

# Build RegistrationCode -> age(years) mapping robustly
if "RegistrationCode" in diet_mb.columns:
    age_by_rc = diet_mb.copy()
    age_by_rc["age"] = pd.to_numeric(age_by_rc["age"], errors="coerce")
    age_by_rc = age_by_rc.groupby("RegistrationCode")["age"].median()
else:
    # assume index is RegistrationCode
    if "age" not in diet_mb.columns:
        raise ValueError("diet_mb does not have an 'age' column.")
    age_by_rc = pd.to_numeric(diet_mb["age"], errors="coerce")
    if diet_mb.index.has_duplicates:
        age_by_rc = pd.to_numeric(diet_mb.groupby(level=0)["age"].median(), errors="coerce")

# -------------------------
# Build X,y
# -------------------------
flat_targets = sum(target_phenotypes, [])
exclude_cols = set(flat_targets + BASE_FEATURES + ["RegistrationCode"])
microbial_features = [c for c in df.columns if c not in exclude_cols]

df_target = df[df[PHENOTYPE].notna()].copy()
y = df_target[PHENOTYPE].astype(float)

# --- NEW: overwrite df_target["age"] with RAW age (years) from diet_mb ---
if "RegistrationCode" not in df_target.columns:
    raise ValueError("df_target is missing 'RegistrationCode' column needed for merging raw age.")

df_target["age"] = pd.to_numeric(df_target["RegistrationCode"].map(age_by_rc), errors="coerce")

miss_frac = df_target["age"].isna().mean()
if miss_frac > 0:
    print(f"[Warning] raw age missing for {miss_frac*100:.1f}% of rows after mapping from diet_mb. "
          f"Will median-impute for modeling/plotting.")

X = df_target[BASE_FEATURES + microbial_features].copy()
X["sex"] = ensure_numeric_sex(X["sex"])

# Drop rows with unmapped sex or missing y
Xy = pd.concat([X, y.rename("y")], axis=1).dropna(subset=["sex", "y"])
y = Xy["y"]
X = Xy.drop(columns=["y"])

# Impute (includes any missing age after mapping)
X = median_impute(X)

if MAIN_FEATURE not in X.columns:
    raise ValueError(
        f"MAIN_FEATURE='{MAIN_FEATURE}' not found in X columns. "
        f"Example columns: {list(X.columns)[:10]} ..."
    )

# -------------------------
# Train LightGBM
# -------------------------
n_estimators = 300 if len(y) < 3000 else 800
learning_rate = 0.05 if len(y) < 3000 else 0.01

model = LGBMRegressor(
    objective="regression",
    n_estimators=n_estimators,
    learning_rate=learning_rate,
    max_depth=3,
    subsample=0.9,
    colsample_bytree=0.8,
    min_child_samples=5,
    n_jobs=8,
    random_state=RANDOM_SEED,
    verbose=-1,
)
model.fit(X, y)

# -------------------------
# Compute SHAP on subsample
# -------------------------
X_used = subsample_df(X, SHAP_SAMPLE_SIZE, RANDOM_SEED)
bg = subsample_df(X_used, min(BACKGROUND_SIZE, len(X_used)), RANDOM_SEED)

explainer = shap.TreeExplainer(model, data=bg, feature_perturbation="interventional")
shap_values = explainer.shap_values(X_used)
if isinstance(shap_values, list):
    shap_values = shap_values[0]
shap_values = np.asarray(shap_values)

# -------------------------
# Plot dependence for ONE feature (TOP histogram + colorbar)
# -------------------------
mpl.rcParams.update(mpl.rcParamsDefault)
plt.rcParams.update({"font.size": FONT})

feat_idx = list(X_used.columns).index(MAIN_FEATURE)
x = X_used[MAIN_FEATURE].to_numpy()
y_shap = shap_values[:, feat_idx]

mask = np.isfinite(x) & np.isfinite(y_shap)

if REMOVE_FLOOR:
    floor_val, frac = infer_floor_value(x)
    if floor_val is not None:
        mask &= (np.round(x, 6) != np.round(floor_val, 6))
        print(f"[{MAIN_FEATURE}] removed inferred floor={floor_val:.6f} (~{100*frac:.1f}%) for plotting")

x = x[mask]
y_plot = y_shap[mask]

# Color-by feature
c = None
c_label = None
if COLOR_BY is not None:
    if COLOR_BY not in X_used.columns:
        raise ValueError(f"COLOR_BY='{COLOR_BY}' not found in X_used columns.")
    c = X_used.loc[X_used.index[mask], COLOR_BY].to_numpy()
    c_label = COLOR_BY

fig = plt.figure(figsize=(FIG_W, FIG_H), dpi=300)
gs = GridSpec(
    2, 2,
    width_ratios=[5.0, 0.22],
    height_ratios=[1.1, 5.0],
    wspace=0.08, hspace=0.05
)

ax_top  = fig.add_subplot(gs[0, 0])
ax_main = fig.add_subplot(gs[1, 0])
cax     = fig.add_subplot(gs[1, 1])

# Main scatter + colorbar
if c is None:
    sc = ax_main.scatter(x, y_plot, s=14, alpha=0.55, edgecolor="none")
    cax.axis("off")
else:
    sc = ax_main.scatter(x, y_plot, c=c, s=14, alpha=0.6, edgecolor="none", cmap="coolwarm")
    cb = fig.colorbar(sc, cax=cax)
    if c_label == "age":
        cb.set_label("Age (years)", rotation=90, labelpad=10)
    else:
        cb.set_label(f"{c_label} (feature value)", rotation=90, labelpad=2)
    cb.ax.tick_params(pad=2)

# Trend line
rm = running_median_line(x, y_plot, n_bins=45, min_per_bin=35)
if rm is not None:
    xm, ym = rm
    ax_main.plot(xm, ym, linewidth=2.2)

# Robust padded limits
xlim = robust_limits(x, pctl=X_PCTL, pad_frac=PAD_FRAC)
ylim = robust_limits(y_plot, pctl=Y_PCTL, pad_frac=PAD_FRAC)
if xlim:
    ax_main.set_xlim(*xlim)
if ylim:
    ax_main.set_ylim(*ylim)

# Labels
ax_main.set_xlabel("B. longum abundance (z-score)")
ax_main.set_ylabel("SHAP value for VAT mass", labelpad=8)

# Secondary top x-axis: percentile mapping
xs = np.sort(x)
ps = np.linspace(0, 100, len(xs))

def x_to_pct(v):
    return np.interp(v, xs, ps)

def pct_to_x(p):
    return np.interp(p, ps, xs)

secax = ax_main.secondary_xaxis("top", functions=(x_to_pct, pct_to_x))
secax.set_xlabel("")                 # place manually
secax.set_xticks([10, 50, 90])

# Top histogram
ax_top.hist(x, bins=TOP_HIST_BINS, density=True, alpha=TOP_HIST_ALPHA)
ax_top.set_xlim(ax_main.get_xlim())
ax_top.axis("off")
ax_top.patch.set_alpha(0.0)

# Layout then place "Percentile"
fig.tight_layout()

top_bbox = ax_top.get_position()
x_center = 0.5 * (top_bbox.x0 + top_bbox.x1)

if str(PCTL_LABEL_POS).lower().startswith("above"):
    y_text = top_bbox.y1 + PCTL_LABEL_DY
    va = "bottom"
else:
    y_text = top_bbox.y0 - PCTL_LABEL_DY
    va = "top"

fig.text(x_center, y_text, "Percentile", ha="center", va=va, fontsize=FONT)

safe_pheno = sanitize_filename(PHENOTYPE)
safe_feat = sanitize_filename(MAIN_FEATURE)
out_png = os.path.join(OUT_DIR, f"{safe_pheno}__dependence__{safe_feat}.png")
out_pdf = os.path.join(OUT_DIR, f"{safe_pheno}__dependence__{safe_feat}.pdf")

fig.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
fig.savefig(out_pdf, dpi=300, bbox_inches="tight", facecolor="white")

plt.show()
plt.close(fig)

print(f"Saved:\n  {out_png}\n  {out_pdf}")
